# Business Entity Resolution — end-to-end competition notebook

**Goal.** For every Source‑1 (reference) entity, return the Source‑2/Source‑3 records that refer to the same business,
or nothing. The metric is **macro F0.5 at the S1-entity level**, so precision counts more than recall, and
a correct "no match" earns full credit.

The notebook answers four questions in order. Every analysis cell ends in a concrete pipeline decision.

1. **What does the data say about the problem?** Integrity, source noise, ground-truth structure, singletons (§1–§5).
2. **Which representations and candidate generator give the most recall for a manageable volume?** (§6–§7, §10–§11)
3. **Which features separate true matches from *hard* negatives?** (§8–§9, §14–§15)
4. **Which decision rule maximises macro F0.5?** (§12–§13, §16–§18)

Then the frozen pipeline runs on the test set and writes `matching_results.tsv` and `candidate_pairs.tsv` (§19–§23).

```text
RAW TSV → normalisation (translit, legal forms, learned token maps) → multi-pass blocking (keys + TF-IDF + GPU encoder)
       → pair features (fuzzy, TF-IDF, numeric, rarity, group context) → GBDT scoring → tuned per-source / anchor
       thresholds → exclusivity (one S1 per S2/S3 record) → matches ⊆ candidates → validated submission
```

**Rules respected:** only competition files are used. There is no internet, no external lookup and no geocoding.
Countries are never hard-coded (France is only in test). Matching is one-to-many, and singletons are never forced to match.

Every tunable knob is in `CFG` (next cell). The defaults are sized for a Kaggle P100 session (≈29 GB RAM, 4 CPUs, 16 GB GPU).

### How to run on Kaggle
1. **Add Data** → attach the competition dataset (any folder name; the notebook finds the 7 TSVs under `/kaggle/input`).
2. **Settings** → Accelerator **GPU P100**. Internet can stay **off**, because nothing is downloaded.
3. **Run All** (or *Save Version → Save & Run All*). Expect a few hours; per-step timings are printed and summarised in §22.
4. Outputs: `/kaggle/working/matching_results.tsv` and `/kaggle/working/candidate_pairs.tsv` (hard-linked into `/kaggle/working/output/`), plus
   `artifacts/` (models, encoder, experiment log, findings) and `Documentation_template_filled.md` (a pre-filled methodology write-up).

If memory or time is tight, lower `N_S1_SAMPLE` (the experiment sample), `TOPK_MAX` or `FEAT_CHUNK`, or set `RUN_CATBOOST` / `RUN_STAGE2` to False.

### Where the knowledge-base / EDA-report insights are used
| insight | where |
|---|---|
| 0 cross-country links → partition by country | all blocking passes (§10), no country feature |
| each S2/S3 record belongs to at most one S1 | exclusivity option in the decision rule (§17) |
| inverted-index stop-word cap on `llc`, `road`, `delhi`… | document-frequency cap in TF-IDF retrieval (§7b/§10) |
| name ∪ address token blocking reaches ~100% recall | passes C/D/F + greedy union (§10) |
| postal codes agree in ~98% of true pairs | postal match/conflict features, pass H (§7, §8) |
| "safe pruning zone" (name and address both dissimilar) | validated pruning (§12) |
| 24% of true pairs have reordered tokens | token-sort/set and set-equality features (§8) |
| within-S1 relative scores | rank / gap / z-score context features (§8) |
| 80% of S1 have both S2 and S3 matches (cross-source confirmation) | stage-2 reranker (§15b) |
| entity-grouped validation, never pair-level | fit/tune/hold by S1 entity (§12) |
| official validator | embedded and executed (§21) |
| licence: MIT/Apache ≤ 8B parameters | only our own small encoder is trained, with no pretrained weights (§10) |

In [ ]:
# =====================================================================================
# 0. SETUP — imports, configuration, utilities
# =====================================================================================
import os, sys, gc, re, csv, json, math, time, random, pickle, unicodedata, warnings, subprocess
from collections import Counter
import multiprocessing as mp

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize as sk_normalize
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 80)
plt.rcParams['figure.dpi'] = 100

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

try:
    display
except NameError:          # running outside Jupyter
    display = print


def _opt_import(name):
    try:
        return __import__(name)
    except Exception as e:  # noqa
        print(f'  [optional] {name} not available: {type(e).__name__}: {e}')
        return None


psutil = _opt_import('psutil')
lgb = _opt_import('lightgbm')
xgb = _opt_import('xgboost')
catboost = _opt_import('catboost')
torch = _opt_import('torch')

try:
    from rapidfuzz import fuzz
    from rapidfuzz.distance import Levenshtein as RF_Lev, JaroWinkler as RF_JW
    try:
        from rapidfuzz.process import cpdist          # vectorised pairwise scorer (rapidfuzz >= 3.6)
    except ImportError:
        cpdist = None
    HAS_RF = True
except ImportError:
    HAS_RF, cpdist = False, None

try:
    import numba
    HAS_NUMBA = True
except Exception:
    HAS_NUMBA = False

try:
    import pyarrow  # noqa
    STR = 'string[pyarrow]'   # ~3x less RAM than python objects for 25M strings
except ImportError:
    STR = object

USE_GPU = bool(torch is not None and torch.cuda.is_available())
if torch is not None:
    torch.manual_seed(SEED)
    if USE_GPU:
        torch.cuda.manual_seed_all(SEED)
if not HAS_RF:
    class _NoRF:                      # fuzzy scorers become NaN features (model still trains on the rest)
        def __getattr__(self, k):
            return None
    fuzz = RF_Lev = RF_JW = _NoRF()
N_JOBS = max(1, min(os.cpu_count() or 1, 8))

CFG = dict(
    # ---------- IO ----------
    LOCAL_ROOTS=['./dataset', '../dataset', '.'],            # used only when /kaggle/input does not exist
    WORK_DIR='/kaggle/working' if os.path.isdir('/kaggle/working') else './working',
    # ---------- experiment scale ----------
    N_S1_SAMPLE=150_000,          # train S1 entities used for blocking/model experiments (entity-level sample)
    FOLD_PCTS=(60, 20, 20),       # fit / tune / hold (by S1 entity hash)
    EDA_SAMPLE=300_000,
    POS_SAMPLE=150_000,           # positive pairs for representation experiments
    QUICK_S1=60_000,              # fit S1 used for ablation models (E2/E3/E7/LOCO)
    # ---------- hashing / retrieval ----------
    HASH_BITS=22, CHAR_BITS=20,
    DF_CAP_FRAC=0.0008, DF_CAP_MIN=2000,   # tokens more frequent than this are ignored by the retrieval passes
    RARE_DF=50,
    SKEL_WEIGHT=0.5, NUM_WEIGHT=1.0, BIGRAM_WEIGHT=1.0, ALPHA_NAME=0.6,
    TOPK_MAX=30, K_GRID=[1, 2, 3, 5, 8, 10, 12, 15, 20, 25, 30], K_RECALL_KEEP=0.995,
    RETR_CHUNK=1000, MAX_BLOCK_POOL=300, GREEDY_MIN_GAIN=0.0005,
    # ---------- features / models ----------
    FEAT_CHUNK=1_000_000,
    LGB_ROUNDS=2500, LGB_LR=0.06, EARLY_STOP=100, QUICK_ROUNDS=600,
    MAX_TRAIN_ROWS=12_000_000,
    PRUNE_MAX_RECALL_LOSS=0.001,  # 'safe pruning zone' is applied only if it loses <= 0.1% of true pairs
    RUN_STAGE2=True, STAGE2_MIN_GAIN=0.0005, STAGE2_KEEP_FEATS=12,
    RUN_XGB=True, RUN_CATBOOST=True, RUN_LR=True,
    # ---------- GPU encoder (learned char-n-gram embeddings; trained ONLY on competition train data) ----------
    RUN_ENCODER=True, ENC_BUCKET_BITS=18, ENC_DIM=64, ENC_EPOCHS=10, ENC_BATCH=1024, ENC_TAU=0.05, ENC_LR=0.01,
    DENSE_CHUNK=256, EMB_MIN_GAIN=0.001,
)
os.makedirs(CFG['WORK_DIR'], exist_ok=True)
ART_DIR = os.path.join(CFG['WORK_DIR'], 'artifacts')
os.makedirs(ART_DIR, exist_ok=True)

T0 = time.time()
TIMINGS, FINDINGS, EXPERIMENTS = [], [], []


def rss_gb():
    return psutil.Process().memory_info().rss / 1e9 if psutil else float('nan')


def log(*msg):
    print(f'[{time.time() - T0:7.0f}s | RSS {rss_gb():5.1f} GB]', *msg, flush=True)


class Timer:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        log(f'>> {self.name}')
        return self

    def __exit__(self, *a):
        dt = time.time() - self.t
        TIMINGS.append(dict(step=self.name, seconds=round(dt, 1), rss_gb=round(rss_gb(), 2)))
        log(f'<< {self.name}: {dt:.1f}s')


def finding(title, evidence, why, action):
    """Every important analytical result is logged as Finding → Evidence → Why → Action."""
    FINDINGS.append(dict(finding=title, evidence=evidence, why=why, action=action))
    print(f'\n■ FINDING : {title}\n  Evidence: {evidence}\n  Why     : {why}\n  Action  : {action}')


def add_experiment(exp, change, m, decision):
    EXPERIMENTS.append(dict(Experiment=exp, Change=change, F05=round(m['F05'], 4),
                            Precision=round(m['P_micro'], 4), Recall=round(m['R_micro'], 4), Decision=decision))


# ---------------- light-weight parallel map (fork on Linux/Kaggle; serial elsewhere) ----------------
def _apply_chunk(args):
    func, items = args
    return [func(x) for x in items]


def run_parallel(fn, args_list, n_jobs=N_JOBS):
    if n_jobs <= 1 or len(args_list) < 2:
        return [fn(a) for a in args_list]
    try:
        ctx = mp.get_context('fork')
    except ValueError:
        return [fn(a) for a in args_list]
    with ctx.Pool(min(n_jobs, len(args_list))) as pool:
        return pool.map(fn, args_list, chunksize=1)


def pmap(func, items, chunk=100_000):
    items = list(items)
    if len(items) < 2 * chunk:
        return [func(x) for x in items]
    parts = run_parallel(_apply_chunk, [(func, items[i:i + chunk]) for i in range(0, len(items), chunk)])
    return [y for part in parts for y in part]


def umap(ser, func, as_str=True):
    """Apply a python function to the UNIQUE values of a Series (parallel), broadcast back."""
    codes, uniq = pd.factorize(ser, sort=False)
    res = pmap(func, uniq.tolist())
    arr = np.empty(len(res), dtype=object)
    arr[:] = res
    out = arr[codes]
    return pd.Series(out, index=ser.index, dtype=STR if as_str else None)


print(f'python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | cpus={os.cpu_count()} '
      f'| GPU={USE_GPU} | rapidfuzz={HAS_RF} (cpdist={cpdist is not None}) | numba={HAS_NUMBA} '
      f'| lightgbm={lgb is not None} | xgboost={xgb is not None} | catboost={catboost is not None} | str dtype={STR}')
if USE_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
if not HAS_RF:
    print('WARNING: rapidfuzz missing -> fuzzy features fall back to slow difflib / NaN.')

## 1. Dataset discovery
The notebook searches `/kaggle/input/` recursively for the seven required TSV files, so the dataset folder name does not matter.
It also looks for a submission validator script. Files are read as **tab-separated**, with `keep_default_na=False` so literal strings like `"null"`/`"NA"` stay visible for the integrity checks.

In [ ]:
REQUIRED = {
    'train_s1': 'train_source1.tsv', 'train_s2': 'train_source2.tsv', 'train_s3': 'train_source3.tsv',
    'train_gt': 'train_ground_truth.tsv',
    'test_s1': 'test_source1.tsv', 'test_s2': 'test_source2.tsv', 'test_s3': 'test_source3.tsv',
}
SOURCE_COLS = ['entity_id', 'business_name', 'business_address', 'country']
GT_COLS = ['source1_entity_id', 'matched_entity_ids']


def discover_files():
    if os.path.isdir('/kaggle/input'):
        roots = ['/kaggle/input']
        print('/kaggle/input contains:', sorted(os.listdir('/kaggle/input')))
    else:
        roots = [r for r in CFG['LOCAL_ROOTS'] if os.path.isdir(r)][:1]
        print('No /kaggle/input — falling back to local roots', roots)
    wanted = {v.lower(): k for k, v in REQUIRED.items()}
    found, validators, tsvs = {}, [], []
    for root in roots:
        for dp, dns, fns in os.walk(root):
            dns[:] = sorted(d for d in dns if not d.startswith('.') and d not in ('working', '__pycache__'))
            for fn in sorted(fns):
                p, low = os.path.join(dp, fn), fn.lower()
                if low.endswith('.tsv'):
                    tsvs.append(p)
                if low in wanted:
                    found.setdefault(wanted[low], []).append(p)
                if low.endswith('.py') and ('valid' in low or 'check' in low or 'eval' in low):
                    validators.append(p)
    print(f'\nTSV files found ({len(tsvs)}):')
    for p in tsvs:
        print(f'   {p}  ({os.path.getsize(p) / 1e6:,.1f} MB)')
    missing = [REQUIRED[k] for k in REQUIRED if k not in found]
    if missing:
        raise FileNotFoundError(f'Missing required competition files: {missing}. Searched roots {roots}. '
                                f'Attach the competition dataset to the notebook.')
    for k, v in found.items():
        if len(v) > 1:
            print(f'   WARNING: {REQUIRED[k]} found {len(v)} times, using {v[0]}')
    return {k: v[0] for k, v in found.items()}, validators


def load_tsv(path, usecols=None):
    df = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, na_filter=False,
                     usecols=usecols, encoding='utf-8')
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(STR)
    return df


def describe_frame(name, df):
    print(f'\n--- {name}: shape={df.shape}  memory={df.memory_usage(deep=True).sum() / 1e6:,.0f} MB')
    print('    dtypes:', {c: str(t) for c, t in df.dtypes.items()})
    display(df.head(3))


with Timer('discover + load all files'):
    PATHS, VALIDATORS = discover_files()
    print('\nValidator-like scripts:', VALIDATORS or 'none found')
    RAW = {k: load_tsv(p) for k, p in PATHS.items()}

for k in REQUIRED:
    need = GT_COLS if k == 'train_gt' else SOURCE_COLS
    miss = [c for c in need if c not in RAW[k].columns]
    if miss:
        raise ValueError(f'{PATHS[k]} is missing columns {miss}; found {list(RAW[k].columns)}')
    describe_frame(k, RAW[k])
print('\nAll 7 required files present with expected columns.')

## 2. Data integrity
These checks look for duplicate IDs, duplicate records, empty or whitespace-only fields, placeholders (`null`, `N/A`, …), malformed IDs and wrong
source prefixes. They also flag countries unseen in training and every ground-truth inconsistency: duplicates, invalid references, and IDs mapped to several S1 entities.

In [ ]:
PLACEHOLDERS = {'null', 'none', 'nan', 'n/a', 'na', 'nil', 'unknown', '-', '--', '0', '.', 'not available', 'tbd'}
TRAIN_COUNTRIES = set(RAW['train_s1'].country.str.strip().unique().tolist())


def integrity_report(name, df, prefix):
    r = dict(file=name, rows=len(df))
    ids = df.entity_id
    r['dup_ids'] = int(ids.duplicated().sum())
    r['dup_full_rows'] = int(df.duplicated().sum())
    r['dup_name_addr_country'] = int(df.duplicated(['business_name', 'business_address', 'country']).sum())
    for col, short in [('business_name', 'name'), ('business_address', 'addr'), ('country', 'ctry')]:
        s = df[col]
        st = s.str.strip()
        r[f'{short}_empty'] = int((s == '').sum())
        r[f'{short}_ws_only'] = int(((st == '') & (s != '')).sum())
        r[f'{short}_placeholder'] = int(st.str.lower().isin(PLACEHOLDERS).sum())
    r['addr_contains_null_token'] = int(df.business_address.str.lower().str.contains(r'\b(?:null|none|nan|n/a)\b').sum())
    r['malformed_ids'] = int((~ids.str.match(r'^S[123]-\d+$')).sum())
    r['wrong_prefix'] = int((~ids.str.startswith(prefix + '-')).sum())
    cs = df.country.str.strip().value_counts()
    r['countries'] = {str(k): int(v) for k, v in cs.items()}
    r['unseen_countries'] = sorted(set(cs.index.tolist()) - TRAIN_COUNTRIES)
    return r


def gt_integrity(gt, s1_ids, s2_ids, s3_ids):
    r = dict(rows=len(gt))
    r['dup_s1_rows'] = int(gt.source1_entity_id.duplicated().sum())
    r['gt_s1_not_in_source1'] = int((~gt.source1_entity_id.isin(s1_ids)).sum())
    r['source1_missing_from_gt'] = int((~s1_ids.isin(gt.source1_entity_id)).sum())
    ex = gt.matched_entity_ids.astype(object).str.split(',').explode()
    raw_nonempty_rows = gt.matched_entity_ids.str.strip() != ''
    r['empty_rows(singletons)'] = int((~raw_nonempty_rows).sum())
    ex = ex[ex.notna()]
    tok = ex.str.strip()
    nonempty_row = raw_nonempty_rows.reindex(tok.index).to_numpy(bool)
    r['empty_tokens_in_nonempty_rows'] = int(((tok == '') & nonempty_row).sum())
    r['ids_with_whitespace'] = int(((ex != tok) & (tok != '')).sum())
    tok = tok[tok != '']
    r['bad_prefix_or_format'] = int((~tok.str.match(r'^S[23]-\d+$')).sum())
    r['invalid_refs'] = int((~(tok.isin(s2_ids) | tok.isin(s3_ids))).sum())
    pairs = pd.DataFrame({'row': tok.index.to_numpy(), 'id': tok.to_numpy()})
    r['dup_within_row'] = int(pairs.duplicated().sum())
    pairs = pairs.drop_duplicates()
    r['ids_mapped_to_multiple_s1'] = int(pairs.id.duplicated().sum())
    r['total_matched_ids'] = int(len(pairs))
    return r


with Timer('integrity checks'):
    rep = [integrity_report(k, RAW[k], 'S' + k[-1]) for k in ['train_s1', 'train_s2', 'train_s3', 'test_s1', 'test_s2', 'test_s3']]
    integ = pd.DataFrame(rep).set_index('file')
    display(integ.drop(columns=['countries']))
    display(pd.DataFrame({k: v['countries'] for k, v in zip(integ.index, rep)}).fillna(0).astype(int))
    GT_INTEG = gt_integrity(RAW['train_gt'], RAW['train_s1'].entity_id, RAW['train_s2'].entity_id, RAW['train_s3'].entity_id)
    display(pd.Series(GT_INTEG, name='ground truth').to_frame())

new_c = sorted(set(c for r in rep for c in r['unseen_countries']))
bad = integ[['dup_ids', 'malformed_ids', 'wrong_prefix']].sum().sum() + GT_INTEG['invalid_refs'] + GT_INTEG['dup_s1_rows']
finding('ID / reference integrity',
        f"dup ids + malformed + wrong prefix + invalid GT refs + dup GT rows = {int(bad)}; "
        f"GT ids mapped to >1 S1 = {GT_INTEG['ids_mapped_to_multiple_s1']}",
        'Broken keys would silently corrupt labels, candidate recall and the submission.',
        'IDs are used as-is (string keys). GT is exploded once into a pair table. '
        + ('Each S2/S3 id belongs to at most one S1 → exclusivity constraint is available to the decision rule.'
           if GT_INTEG['ids_mapped_to_multiple_s1'] == 0 else 'Some ids map to several S1 → exclusivity NOT enforced.'))
finding('Countries unseen in training', f'{new_c} appear only in test',
        'A model with country-specific features/thresholds would be undefined on these rows.',
        'No country feature, no hard-coded country list; blocking is done *within* the (data-driven) country value; '
        'transliteration & legal-form handling are language-agnostic.')
finding('Missing / placeholder addresses',
        f"empty addr: " + ', '.join(f"{i}={integ.loc[i, 'addr_empty'] / integ.loc[i, 'rows']:.2%}" for i in integ.index)
        + f"; addr containing null/N/A tokens: " + ', '.join(f"{i}={integ.loc[i, 'addr_contains_null_token'] / integ.loc[i, 'rows']:.2%}" for i in integ.index),
        'Address similarity is undefined for these records; placeholder tokens create fake agreement.',
        'Placeholder tokens are removed during normalisation; missingness flags are model features; name-only retrieval passes exist.')

## 3. Deep EDA — source-specific noise
Each source is profiled on its full data (counts, uniqueness, duplicates) and on a sample (string shape, scripts, noise signatures).
Test sources are profiled too, to measure the train→test shift. Every statistic below feeds a normalisation, feature or blocking decision.

In [ ]:
_DOMAIN_RE = re.compile(r'\.(?:com|net|org|in|co|biz|info|io|us|fr)\b|^[@#][a-z0-9]|www\.')
_NULL_TOK_RE = r'\b(?:null|none|nan|n/a)\b'


def script_of(s):
    for ch in s:
        if ord(ch) > 0x24F and ch.isalpha():
            return unicodedata.name(ch, 'OTHER X').split(' ')[0]
    return 'LATIN'


def source_profile(name, df, n_sample):
    r = dict(source=name, records=len(df))
    r['uniq_names_%'] = 100 * df.business_name.nunique() / len(df)
    r['uniq_addr_%'] = 100 * df.business_address.nunique() / len(df)
    r['n_countries'] = df.country.nunique()
    r['empty_addr_%'] = 100 * (df.business_address.str.strip() == '').mean()
    r['dup_record_%'] = 100 * df.duplicated(['business_name', 'business_address', 'country']).mean()
    smp = df.sample(min(n_sample, len(df)), random_state=SEED)
    nm, ad = smp.business_name.astype(object), smp.business_address.astype(object)
    nlen, alen = nm.str.len(), ad.str.len()
    r['name_len_med'], r['name_len_p95'] = nlen.median(), nlen.quantile(.95)
    r['name_words_mean'] = nm.str.split().str.len().mean()
    r['name_digit_%'] = 100 * nm.str.contains(r'\d').mean()
    r['name_punct_%'] = 100 * nm.str.contains(r'[^\w\s]').mean()
    r['name_nonascii_%'] = 100 * (~nm.map(str.isascii)).mean()
    r['name_accented_latin_%'] = 100 * nm.str.contains('[À-ɏ]').mean()
    r['name_upper_%'] = 100 * nm.map(str.isupper).mean()
    r['name_junk_prefix_%'] = 100 * nm.map(lambda s: bool(s) and not s[0].isalnum()).mean()
    r['name_domain_handle_%'] = 100 * nm.str.lower().map(lambda s: bool(_DOMAIN_RE.search(s))).mean()
    r['addr_len_med'] = alen.median()
    r['addr_words_mean'] = ad.str.split().str.len().mean()
    r['addr_digits_mean'] = ad.str.count(r'\d').mean()
    r['addr_null_token_%'] = 100 * ad.str.lower().str.contains(_NULL_TOK_RE).mean()
    r['addr_nonascii_%'] = 100 * (~ad.map(str.isascii)).mean()
    scripts = nm.map(script_of).value_counts(normalize=True)
    return r, scripts, nlen.clip(upper=100), alen.clip(upper=200)


with Timer('source profiling'):
    PROF, SCRIPTS, NLEN, ALEN = {}, {}, {}, {}
    for k in ['train_s1', 'train_s2', 'train_s3', 'test_s1', 'test_s2', 'test_s3']:
        PROF[k], SCRIPTS[k], NLEN[k], ALEN[k] = source_profile(k, RAW[k], CFG['EDA_SAMPLE'])
    prof = pd.DataFrame(PROF).T.drop(columns='source')
    display(prof.astype(float).round(2).T)

fig, ax = plt.subplots(1, 4, figsize=(22, 4.2))
for k in ['train_s1', 'train_s2', 'train_s3']:
    ax[0].hist(NLEN[k], bins=60, histtype='step', density=True, label=k)
    ax[1].hist(ALEN[k], bins=60, histtype='step', density=True, label=k)
ax[0].set_title('name length (chars)'); ax[1].set_title('address length (chars)')
ax[0].legend(); ax[1].legend()
noise_cols = ['name_upper_%', 'name_junk_prefix_%', 'name_domain_handle_%', 'name_nonascii_%',
              'name_accented_latin_%', 'empty_addr_%', 'addr_null_token_%']
prof.loc[['train_s1', 'train_s2', 'train_s3'], noise_cols].astype(float).T.plot.bar(ax=ax[2], rot=60)
ax[2].set_title('noise signatures by source (% records)')
sc = pd.DataFrame({k: SCRIPTS[k] for k in ['train_s1', 'train_s2', 'train_s3', 'test_s2']}).fillna(0)
sc = sc.loc[sc.max(axis=1) > 0.001]
(1 - sc.loc[['LATIN']]).T.rename(columns={'LATIN': 'non-Latin script share'}).plot.bar(ax=ax[3], rot=0, legend=False)
ax[3].set_title('share of names in non-Latin scripts')
plt.tight_layout(); plt.show()
display((sc * 100).round(2))

cc = pd.DataFrame({k: RAW[k].country.str.strip().value_counts(normalize=True) for k in
                   ['train_s1', 'train_s2', 'train_s3', 'test_s1', 'test_s2', 'test_s3']}).fillna(0)
display((cc * 100).round(2))

P_ = prof.astype(float)
finding('Source-specific casing noise',
        f"ALL-CAPS names: S1 {P_.loc['train_s1', 'name_upper_%']:.1f}% vs S2 {P_.loc['train_s2', 'name_upper_%']:.1f}% vs S3 {P_.loc['train_s3', 'name_upper_%']:.1f}%",
        'Case differences break exact matching and inflate edit distance.',
        'Case-fold everything; keep one raw-exact feature so the model can still reward identical raw strings.')
nonlat = {k: 100 * (1 - SCRIPTS[k].get('LATIN', 0)) for k in SCRIPTS}
finding('Non-Latin scripts (transliteration noise)',
        'non-Latin names: ' + ', '.join(f'{k}={v:.1f}%' for k, v in nonlat.items()),
        'S1 is Latin-only; a Devanagari/Telugu/... S2/S3 name shares no characters with its S1 entity.',
        'Rule-based, dependency-free transliteration (Unicode character names → Latin) + consonant-skeleton keys; '
        'the model receives a non-ASCII flag.')
finding('Junk prefixes, domains and handles',
        f"junk prefix S2/S3 = {P_.loc['train_s2', 'name_junk_prefix_%']:.1f}%/{P_.loc['train_s3', 'name_junk_prefix_%']:.1f}%, "
        f"domain/handle S2/S3 = {P_.loc['train_s2', 'name_domain_handle_%']:.1f}%/{P_.loc['train_s3', 'name_domain_handle_%']:.1f}%",
        '"victorylaboratories.com" / "@memorialproject" have no token overlap with "Victory Laboratories Limited".',
        'Strip punctuation/TLDs; add a *squashed* (space-free) name representation used as a blocking key and fuzzy feature.')
finding('Common (non-unique) business names in the reference source',
        f"only {P_.loc['train_s1', 'uniq_names_%']:.1f}% of S1 names are unique strings",
        'Many distinct entities share a name → name-only matching produces false merges (precision is weighted 2x in F0.5).',
        'Name-frequency (commonness) features on both sides; address agreement required by the model for common names.')
finding('Train→test shift',
        'max |Δ| (train vs test S2) over profile stats: ' +
        ', '.join(f"{c}={abs(P_.loc['train_s2', c] - P_.loc['test_s2', c]):.2f}" for c in noise_cols[:5]),
        'Large shifts would invalidate thresholds tuned on train.',
        'Noise signatures are similar; the only structural shift is the new country (handled by country-agnostic design).')
TEST_COUNTS = {k: len(RAW[k]) for k in ['test_s1', 'test_s2', 'test_s3']}
for k in ['test_s1', 'test_s2', 'test_s3']:
    del RAW[k]                       # reloaded in §20 — frees memory for the training phase
gc.collect()

## 4. Ground-truth reconstruction
`matched_entity_ids` is exploded into a pair table `(S1 entity, S2/S3 entity, source pair, target=1)`.
The pool of S2 and S3 records is built and sorted by country so that every country is a contiguous slice (used by blocking).
The **entity-level split** used everywhere is defined here as well (fit/tune/hold by a hash of the S1 id; see §12).

In [ ]:
def build_ground_truth(gt):
    ex = gt[GT_COLS].astype(object).copy()
    ex['o_id'] = ex.matched_entity_ids.str.split(',')
    ex = ex.explode('o_id')
    ex['o_id'] = ex.o_id.str.strip()
    ex = ex[ex.o_id.notna() & (ex.o_id != '')]
    pairs = pd.DataFrame({'s1_id': ex.source1_entity_id.to_numpy(), 'o_id': ex.o_id.to_numpy()}).drop_duplicates()
    pairs['src'] = pairs.o_id.str[:2]
    pairs['target'] = np.int8(1)
    return pairs.reset_index(drop=True)


def build_pool(s2, s3):
    s2 = s2.copy(); s2['src'] = np.int8(2)
    s3 = s3.copy(); s3['src'] = np.int8(3)
    P = pd.concat([s2, s3], ignore_index=True)
    P['country_key'] = P.country.str.strip().str.lower()
    return P.sort_values('country_key', kind='stable').reset_index(drop=True)


with Timer('ground truth + pool'):
    S1 = RAW.pop('train_s1')
    S1['country_key'] = S1.country.str.strip().str.lower()
    P = build_pool(RAW.pop('train_s2'), RAW.pop('train_s3'))
    gt_pairs = build_ground_truth(RAW.pop('train_gt'))
    gc.collect()
    gt_pairs['s1_idx'] = pd.Index(S1.entity_id.astype(object)).get_indexer(gt_pairs.s1_id).astype(np.int32)
    _pidx = pd.Index(P.entity_id.astype(object))
    gt_pairs['pi'] = _pidx.get_indexer(gt_pairs.o_id).astype(np.int32)
    del _pidx
    bad_ref = int(((gt_pairs.s1_idx < 0) | (gt_pairs.pi < 0)).sum())
    if bad_ref:
        print(f'WARNING: dropping {bad_ref} GT pairs with unknown ids')
        gt_pairs = gt_pairs[(gt_pairs.s1_idx >= 0) & (gt_pairs.pi >= 0)].reset_index(drop=True)
    NS1, NP = len(S1), len(P)

N_TRUE_ALL = np.bincount(gt_pairs.s1_idx, minlength=NS1)
N_S2 = np.bincount(gt_pairs.s1_idx[gt_pairs.src == 'S2'], minlength=NS1)
N_S3 = np.bincount(gt_pairs.s1_idx[gt_pairs.src == 'S3'], minlength=NS1)
print(f'S1={NS1:,}  pool(S2+S3)={NP:,}  GT pairs={len(gt_pairs):,}')
display(gt_pairs.head())

mult = pd.Series(N_TRUE_ALL).value_counts().sort_index()
share0, share1, share_many = (N_TRUE_ALL == 0).mean(), (N_TRUE_ALL == 1).mean(), (N_TRUE_ALL > 1).mean()
p_src = P.src.to_numpy()
matched_mask = np.zeros(NP, bool); matched_mask[gt_pairs.pi.to_numpy()] = True
tab = pd.DataFrame({
    'S1 with >=1 match %': [100 * (N_S2 > 0).mean(), 100 * (N_S3 > 0).mean()],
    'mean matches / S1': [N_S2.mean(), N_S3.mean()],
    'max matches / S1': [N_S2.max(), N_S3.max()],
    'pool records matched to some S1 %': [100 * matched_mask[p_src == 2].mean(), 100 * matched_mask[p_src == 3].mean()],
}, index=['S1-S2', 'S1-S3'])
display(tab.round(2))
display(pd.crosstab(pd.Series(N_S2, name='#S2 matches'), pd.Series(N_S3, name='#S3 matches')))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
mult.plot.bar(ax=ax[0], color='steelblue'); ax[0].set_title('matches per S1 entity'); ax[0].set_xlabel('# true matches')
pd.DataFrame({'S2': pd.Series(N_S2).value_counts(normalize=True), 'S3': pd.Series(N_S3).value_counts(normalize=True)}).sort_index().plot.bar(ax=ax[1])
ax[1].set_title('per-source multiplicity (share of S1)')
plt.tight_layout(); plt.show()

kind = 'one-to-many' if share_many > 0.5 else ('one-to-one' if share1 > 0.8 else 'mixed')
finding('Matching cardinality',
        f'singletons {share0:.2%}, exactly one match {share1:.2%}, >1 match {share_many:.2%}; mean {N_TRUE_ALL.mean():.2f}, max {N_TRUE_ALL.max()}; '
        f'S1 with both S2 & S3 matches {((N_S2 > 0) & (N_S3 > 0)).mean():.2%}',
        'Top-1 assignment would cap recall far below 1 for most entities.',
        f'Problem is {kind.upper()}: threshold-based multi-match decisions (never top-1 only); S2 and S3 are *not* internally deduplicated '
        f'(up to {N_S2.max()} S2 records per S1) → the decision rule must accept several records from the same source.')
finding('Pool contains many distractors',
        f"{100 * (1 - matched_mask.mean()):.1f}% of S2/S3 records are not matched to any S1",
        'Candidate generation will surface unmatched look-alikes → need hard-negative training.',
        'Train on blocking candidates (natural hard negatives), not on random pairs (tested in E3).')
finding('Exclusivity of pool records',
        f"{GT_INTEG['ids_mapped_to_multiple_s1']} S2/S3 ids linked to more than one S1",
        'If a record belongs to at most one S1, competing S1 claims can be resolved globally.',
        'Decision rule option "exclusive": each S2/S3 record is kept only for its best-scoring S1 (tuned in §17).')

# ---- entity-level split (fit / tune / hold) + experimental sample ----
_h = pd.util.hash_pandas_object(S1.entity_id.astype(object), index=False).to_numpy() % 100
f1, f2, _ = CFG['FOLD_PCTS']
S1_FOLD = np.where(_h < f1, 'fit', np.where(_h < f1 + f2, 'tune', 'hold'))
gt_pairs['fold'] = S1_FOLD[gt_pairs.s1_idx.to_numpy()]
SAMPLE_IDX = np.sort(np.random.RandomState(SEED).choice(NS1, min(CFG['N_S1_SAMPLE'], NS1), replace=False))
print('Fold sizes (all S1):', pd.Series(S1_FOLD).value_counts().to_dict(),
      '| sample fold sizes:', pd.Series(S1_FOLD[SAMPLE_IDX]).value_counts().to_dict())

### Text-normalisation engine
These functions are defined once and used unchanged by validation and test inference. §6 and §7 quantify each transformation.
* **Transliteration**: the Python standard library `unicodedata` turns Indic, Greek and Cyrillic characters into Latin letters using their Unicode *names*
  (e.g. `DEVANAGARI LETTER TTA → t`, `VOWEL SIGN AA → a`), and strips accents with NFKD. There are no external tables or downloads.
* **Names**: `basic` (case, translit, punctuation, `&→and`, TLD removal, `L.L.C.→llc`) → `core` (legal forms and stop words removed,
  leetspeak repaired, learned synonym map) → `squash` (no spaces) → `skeleton` (consonant skeleton, robust to vowels and typos).
* **Addresses**: `basic` (placeholders removed, ordinals `3rd→3`) → `canon` (multilingual street-type map + learned map from train pairs).

In [ ]:
_SPECIAL = {'œ': 'oe', 'æ': 'ae', 'ß': 'ss', 'ø': 'o', 'ł': 'l', 'đ': 'd', 'ð': 'd', 'þ': 'th', 'ı': 'i', 'ĳ': 'ij',
            '’': "'", '‘': "'", '´': "'", '`': "'", '–': '-', '—': '-', '“': '"', '”': '"', '«': '"', '»': '"',
            '।': ' ', '॥': ' '}
_VOWELS = {'a': 'a', 'aa': 'a', 'i': 'i', 'ii': 'i', 'u': 'u', 'uu': 'u', 'e': 'e', 'ee': 'e',
           'ai': 'ai', 'o': 'o', 'oo': 'o', 'au': 'au'}
_REP = re.compile(r'(.)\1+')


def _indic_letter(rest):
    w = rest.lower().replace('-', ' ').split()
    if not w:
        return ''
    if w[0] == 'vocalic':
        return 'ri' if len(w) > 1 and w[1].startswith('r') else 'li'
    if w[0] in ('candra', 'short', 'independent', 'small', 'capital', 'final', 'medial', 'initial') and len(w) > 1:
        w = w[1:]
    tok = w[-1]
    if tok in _VOWELS:
        return _VOWELS[tok]
    if len(tok) > 1 and tok.endswith('a'):
        tok = tok[:-1]                       # consonant: drop inherent vowel (KA -> k, TTA -> t)
    return _REP.sub(r'\1', tok)


def _char_map(ch):
    if ch in _SPECIAL:
        return _SPECIAL[ch]
    cat = unicodedata.category(ch)
    name = unicodedata.name(ch, '')
    if cat in ('Mn', 'Mc', 'Me', 'Cf'):          # combining marks, Indic vowel signs, ZWJ...
        if 'VOWEL SIGN' in name:
            v = name.split('VOWEL SIGN ', 1)[1].lower().split()
            if v and v[0] == 'vocalic':
                return 'ri'
            return _VOWELS.get(v[-1], '') if v else ''
        if name.endswith('ANUSVARA') or name.endswith('CANDRABINDU'):
            return 'n'
        if name.endswith('VISARGA'):
            return 'h'
        return ''
    if cat == 'Nd':
        d = unicodedata.decimal(ch, None)
        return str(d) if d is not None else ' '
    base = ''.join(c for c in unicodedata.normalize('NFKD', ch) if not unicodedata.combining(c))
    if base and base.isascii():
        return base.lower()
    if cat.startswith('L') and ' LETTER ' in name:
        return _indic_letter(name.split(' LETTER ', 1)[1])
    if cat[0] in 'PSZ':
        return ' '
    return ch


class _TranslitTable(dict):
    def __missing__(self, key):
        v = _char_map(chr(key))
        self[key] = v
        return v


TRANSLIT = _TranslitTable()


def translit(s):
    return s if s.isascii() else s.translate(TRANSLIT)


LEGAL_FORMS = {
    'inc', 'incorporated', 'llc', 'llp', 'pllc', 'lp', 'ltd', 'limited', 'pvt', 'private', 'privet', 'praivet', 'prayvet',
    'corp', 'corporation', 'co', 'cos', 'company', 'plc', 'pte', 'opc', 'gmbh', 'ag', 'sa', 'sas', 'sasu', 'sarl', 'eurl',
    'sci', 'snc', 'scs', 'scop', 'selarl', 'sprl', 'bv', 'nv', 'srl', 'spa',
    'the', 'and', 'of', 'et', 'de', 'du', 'des', 'la', 'le', 'les', 'l', 'd',
}
ADDR_HAND_MAP = {
    'street': 'st', 'str': 'st', 'strt': 'st', 'avenue': 'ave', 'av': 'ave', 'avn': 'ave', 'road': 'rd', 'drive': 'dr',
    'drv': 'dr', 'lane': 'ln', 'court': 'ct', 'crt': 'ct', 'place': 'pl', 'boulevard': 'blvd', 'boul': 'blvd',
    'bd': 'blvd', 'blv': 'blvd', 'highway': 'hwy', 'parkway': 'pkwy', 'pky': 'pkwy', 'circle': 'cir', 'terrace': 'ter',
    'terr': 'ter', 'trail': 'trl', 'square': 'sq', 'suite': 'ste', 'apartment': 'apt', 'appt': 'apt', 'building': 'bldg',
    'floor': 'fl', 'flr': 'fl', 'north': 'n', 'south': 's', 'east': 'e', 'west': 'w', 'northeast': 'ne',
    'northwest': 'nw', 'southeast': 'se', 'southwest': 'sw', 'mount': 'mt', 'fort': 'ft', 'route': 'rte',
    'expressway': 'expy', 'freeway': 'fwy', 'junction': 'jct', 'heights': 'hts', 'center': 'ctr', 'centre': 'ctr',
    'point': 'pt', 'first': '1', 'second': '2', 'third': '3', 'fourth': '4', 'fifth': '5', 'sixth': '6',
    'seventh': '7', 'eighth': '8', 'ninth': '9', 'tenth': '10', 'number': 'no', 'nr': 'near', 'opposite': 'opp',
    'sector': 'sec', 'sect': 'sec', 'district': 'dist', 'r': 'rue', 'chemin': 'ch', 'allee': 'all', 'impasse': 'imp',
    'faubourg': 'fbg', 'saint': 'st', 'sainte': 'ste', 'residence': 'res', 'batiment': 'bat',
}
ADDR_PLACEHOLDER_TOKENS = {'null', 'none', 'nan', 'na', 'nil', 'unknown', 'undefined', 'notavailable'}
NAME_MAP, ADDR_MAP = {}, {}          # learned from TRAIN fit-fold positive pairs in §6/§7 (empty until then)

_WWW = re.compile(r'\bwww\.')
_TLD = re.compile(r'\.(?:com|net|org|in|co|biz|info|io|us|fr|uk|online|site|store|shop)\b')
_APOS = re.compile(r"['’`´]")
_NON_ALNUM = re.compile(r'[^0-9a-z]+')
_SINGLE_RUN = re.compile(r'\b(?:[a-z] )+[a-z]\b')
_ORDINAL = re.compile(r'\b(\d+)(?:st|nd|rd|th)\b')
_NUM = re.compile(r'\d+')
_VOW = re.compile(r'[aeiouy]')
_TRADE = re.compile(r'\b(?:ta|dba|aka|fka|trading as|doing business as)\b')
_LEET = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's', '7': 't'})


def _merge_singles(m):
    return m.group(0).replace(' ', '')


def name_basic(raw):
    s = translit(raw.lower())
    s = _TLD.sub(' ', _WWW.sub(' ', s))
    s = _APOS.sub('', s.replace('&', ' and '))
    s = _NON_ALNUM.sub(' ', s).strip()
    return _SINGLE_RUN.sub(_merge_singles, s)


def _fix_leet(t):
    if t.isalpha() or t.isdigit():
        return t
    if sum(c.isdigit() for c in t) == 1 and len(t) >= 2:
        return t.translate(_LEET)
    return t


def name_core(basic):
    toks = [NAME_MAP.get(t, t) for t in (_fix_leet(t) for t in basic.split())]
    core = [t for t in toks if t not in LEGAL_FORMS]
    return ' '.join(core if core else toks)


def skel_tok(t):
    if not t or t.isdigit():
        return t
    t = t.replace('ph', 'f').replace('ck', 'k').replace('q', 'k').replace('w', 'v').replace('z', 's').replace('sh', 's')
    return _REP.sub(r'\1', t[0] + _VOW.sub('', t[1:]))


def name_skel(core):
    return ' '.join(skel_tok(t) for t in core.split())


def name_alt(basic):
    """Trade-name part after 't/a', 'dba', 'aka' ... (empty if none)."""
    parts = _TRADE.split(basic)
    return name_core(parts[-1].strip()) if len(parts) > 1 and parts[-1].strip() else ''


def acronym(core):
    t = [x for x in core.split() if not x.isdigit()]
    return ''.join(x[0] for x in t) if len(t) >= 2 else ''


def addr_basic(raw):
    s = translit(raw.lower())
    s = _APOS.sub('', s.replace('&', ' and '))
    s = _ORDINAL.sub(r'\1', _NON_ALNUM.sub(' ', s))
    s = _SINGLE_RUN.sub(_merge_singles, s.strip())
    return ' '.join(t for t in s.split() if t not in ADDR_PLACEHOLDER_TOKENS)


def addr_canon(basic):
    return ' '.join(ADDR_MAP.get(t2, t2) for t2 in (ADDR_HAND_MAP.get(t, t) for t in basic.split()))


def addr_keys(canon):
    """house-number + street-token keys, robust to component re-ordering ('OR, Eugene, 3900 River Rd')."""
    t = canon.split()
    keys = []
    for i, x in enumerate(t[:-1]):
        if x.isdigit() and not t[i + 1].isdigit() and len(t[i + 1]) >= 3:
            keys.append((x.lstrip('0') or '0') + '|' + skel_tok(t[i + 1])[:4])
            if len(keys) >= 3:
                break
    return ' '.join(keys)


def name_flags(raw):
    low = raw.lower()
    b = 0
    if _DOMAIN_RE.search(low):
        b |= 1
    if not raw.isascii():
        b |= 2
    if raw.isupper():
        b |= 4
    if raw[:1] and not raw[:1].isalnum():
        b |= 8
    return b


def first_number(canon):
    m = _NUM.search(canon)
    return int(m.group(0)[:9]) if m else -1


def learn_token_map(left, right, min_count=40, min_prob=0.5, max_diff=2):
    """Data-driven synonym map from positive pairs: token r (noisy side) → token l (S1 side) when r systematically
    replaces l. Keys and targets are kept disjoint (no chains)."""
    pair_c, r_c = Counter(), Counter()
    for a, b in zip(left, right):
        A, B = set(a.split()), set(b.split())
        L, R = A - B, B - A
        if not L or not R or len(L) > max_diff or len(R) > max_diff:
            continue
        w = 1.0 / (len(L) * len(R))
        for r in R:
            r_c[r] += 1
            for l in L:
                pair_c[(r, l)] += w
    mp, targets = {}, set()
    for (r, l), c in sorted(pair_c.items(), key=lambda x: -x[1]):
        if c < min_count:
            break
        if r in mp or l in mp or r in targets or r == l or r.isdigit() or l.isdigit():
            continue
        if c / r_c[r] >= min_prob:
            mp[r] = l
            targets.add(l)
    return mp


for s in ['राम मार्केटिंग प्राइवेट लिमिटेड', 'Smt SMB (INDIA) ÉDUCATION PRIVATE LIMITED', 'victorylaboratories.com',
          'Jordan & Hartley Business L.L.C.', 'Community League 0f New Philadelphia Corp', 'Iridova t/a Etti\'s Audio Installation']:
    b = name_basic(s)
    print(f'{s!r:50} basic={b!r:45} core={name_core(b)!r:35} skel={name_skel(name_core(b))!r} alt={name_alt(b)!r}')
for s in ['OR, Eugene, 3900 River Road', '520 3TH STREET, NEW PHILADELPHIA, OH', 'B-4/18 Tyagi Bhavanashok Vihar Ph-ii, North Delhi, दिल्ली',
          '5403 BENNINTGON AVE, NULL, KANSAS CITY, MO']:
    b = addr_basic(s)
    print(f'{s!r:62} basic={b!r:50} canon={addr_canon(b)!r:45} keys={addr_keys(addr_canon(b))!r}')

## 5. Singleton analysis
About 5% of S1 entities have **no** match. Under F0.5 each of them scores 1 when nothing is predicted and 0 otherwise.
This section asks whether singletons can be recognised from the S1 record alone (length, missingness, country, commonness, token rarity).
Candidate-based evidence (candidate counts, best retrieval score) is added in §11, once candidates exist.

In [ ]:
with Timer('singleton analysis'):
    S1['n_basic'] = umap(S1.business_name, name_basic)
    S1['a_basic'] = umap(S1.business_address, addr_basic)
    sg = pd.DataFrame({
        'singleton': N_TRUE_ALL == 0,
        'country': S1.country.astype(object).to_numpy(),
        'name_len': S1.business_name.str.len().to_numpy(np.float32),
        'name_tokens': S1.n_basic.str.count(' ').to_numpy(np.float32) + 1,
        'addr_len': S1.business_address.str.len().to_numpy(np.float32),
        'addr_tokens': S1.a_basic.str.count(' ').to_numpy(np.float32) + 1,
        'addr_missing': (S1.a_basic == '').to_numpy(bool),
        'addr_has_number': S1.a_basic.str.contains(r'\d').to_numpy(bool),
        'name_dup_in_s1': S1.n_basic.map(S1.n_basic.value_counts()).to_numpy(np.float32),
        'addr_dup_in_s1': S1.a_basic.map(S1.a_basic.value_counts()).to_numpy(np.float32),
    })
    _tok = S1.n_basic.astype(object).str.split().explode()
    _tf = _tok.map(_tok.value_counts())
    sg['name_min_token_freq'] = _tf.groupby(level=0).min().reindex(range(NS1)).fillna(0).to_numpy(np.float32)
    sg["has_legal_form"] = _tok.isin(LEGAL_FORMS).groupby(level=0).any().reindex(range(NS1)).fillna(False).astype(bool).to_numpy()
    del _tok, _tf

num_cols = [c for c in sg.columns if c not in ('singleton', 'country')]
cmp = sg.groupby('singleton')[num_cols].agg(['mean', 'median']).T.unstack()
cmp.columns = [f'{"singleton" if a else "matched"}_{b}' for a, b in cmp.columns]
display(cmp.round(3))
display(sg.groupby('country').singleton.agg(['mean', 'size']).rename(columns={'mean': 'singleton_rate'}))

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
b1 = pd.cut(sg.name_dup_in_s1, [0, 1, 2, 5, 20, 1e9], labels=['1', '2', '3-5', '6-20', '>20'])
sg.groupby(b1).singleton.mean().plot.bar(ax=ax[0], rot=0, color='indianred')
ax[0].set_title('singleton rate vs how many S1 share the name'); ax[0].axhline(sg.singleton.mean(), ls='--', c='k')
b2 = pd.qcut(sg.name_min_token_freq.rank(method='first'), 10, labels=False)
sg.groupby(b2).singleton.mean().plot.bar(ax=ax[1], rot=0, color='indianred')
ax[1].set_title('singleton rate by rarest-name-token frequency decile (0 = rarest)'); ax[1].axhline(sg.singleton.mean(), ls='--', c='k')
plt.tight_layout(); plt.show()

# How predictable is "singleton" from S1 attributes alone?
_smp = sg.sample(min(300_000, NS1), random_state=SEED)
_X = _smp[num_cols].astype(np.float32).assign(country=pd.factorize(_smp.country)[0])
_y = _smp.singleton.to_numpy()
_cut = int(len(_X) * 0.7)
if lgb is not None:
    _m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1, random_state=SEED)
    _m.fit(_X.iloc[:_cut], _y[:_cut])
    SINGLETON_AUC = roc_auc_score(_y[_cut:], _m.predict_proba(_X.iloc[_cut:])[:, 1])
else:
    from sklearn.linear_model import LogisticRegression
    _m = LogisticRegression(max_iter=500).fit(_X.iloc[:_cut].fillna(0), _y[:_cut])
    SINGLETON_AUC = roc_auc_score(_y[_cut:], _m.predict_proba(_X.iloc[_cut:].fillna(0))[:, 1])
print(f'S1-attribute-only singleton classifier AUC = {SINGLETON_AUC:.3f}')
finding('Singletons from S1 attributes alone',
        f'singleton rate {sg.singleton.mean():.2%}; attribute-only AUC {SINGLETON_AUC:.3f}; '
        f'rate for names shared by >20 S1 = {sg.loc[sg.name_dup_in_s1 > 20, "singleton"].mean():.2%}',
        'If S1 attributes alone separated singletons we could gate them before matching; a weak AUC means the '
        'no-match decision must come from the (lack of) candidate evidence.',
        'No separate singleton gate. S1-side attributes (name/address commonness, lengths, missingness) are fed to the pair '
        'model, and singletons are handled by the anchor threshold (an S1 gets matches only if its best candidate is '
        'confident) — tuned for macro F0.5 in §16–§17.')
del _smp, _X, _y, _m
gc.collect()

## 6. Name-normalisation experiments
Every transformation is judged on two opposing effects:
* **Useful collisions**: the share of true (S1, S2/S3) pairs whose representations become *equal* (`pos_equal_pct`, a recall proxy).
* **Dangerous collisions**: S1 is a deduplicated reference, so any two S1 records that become equal are *different entities*
  (`S1_collision_pct`). `exact_rule_precision_pct` is the precision of the rule "match every S1 in the same country with an equal representation".

A learned synonym map (e.g. `intl→international`) is estimated **from fit-fold positive pairs only** before these stages run.

In [ ]:
def country_slices(Pdf):
    ck = Pdf.country_key.astype(object).to_numpy()
    b = np.flatnonzero(ck[1:] != ck[:-1]) + 1
    starts, ends = np.r_[0, b], np.r_[b, len(ck)]
    return {ck[s]: (int(s), int(e)) for s, e in zip(starts, ends)}


def random_same_country(ck_arr, slices, rng):
    s = np.array([slices[c][0] for c in ck_arr]); e = np.array([slices[c][1] for c in ck_arr])
    return (s + (rng.rand(len(s)) * (e - s)).astype(np.int64)).astype(np.int64)


# named stage functions (picklable for the parallel map)
def rep_raw(s): return s
def rep_lower(s): return ' '.join(s.lower().split())
def rep_translit(s): return ' '.join(translit(s.lower()).split())
def rep_nbasic(s): return name_basic(s)
def rep_ncore(s): return name_core(name_basic(s))
def rep_nsorted(s): return ' '.join(sorted(name_core(name_basic(s)).split()))
def rep_nsquash(s): return name_core(name_basic(s)).replace(' ', '')
def rep_nskel(s): return name_skel(name_core(name_basic(s))).replace(' ', '')


def rep_experiment(stages, s1_text, s1_ck, pos_s1, o_text, o_ck, s1_raw, o_raw):
    rows, prev_key, prev_eq = [], None, None
    for name, fn in stages:
        r_s1 = umap(s1_text, fn).astype(object).to_numpy()
        r_o = np.empty(len(o_text), dtype=object); r_o[:] = [fn(x) for x in o_text]
        k_s1, k_o = s1_ck + '|' + r_s1, o_ck + '|' + r_o
        eq = (r_s1[pos_s1] == r_o) & (r_o != '')
        vc = pd.Series(k_s1).value_counts()
        n_same = pd.Series(k_o).map(vc).fillna(0).to_numpy()
        n_same[r_o == ''] = 0
        dup = pd.Series(k_s1).duplicated(keep=False).to_numpy()
        rows.append(dict(representation=name, unique_values=len(vc), S1_collision_pct=100 * dup.mean(),
                         pos_equal_pct=100 * eq.mean(), exact_rule_precision_pct=100 * eq.sum() / max(n_same.sum(), 1),
                         mean_S1_sharing_value=n_same.mean()))
        if prev_key is not None:
            for i in np.flatnonzero(eq & ~prev_eq)[:2]:
                print(f'   [{name}] useful collision   : {s1_raw[pos_s1[i]]!r}  ==  {o_raw[i]!r}')
            g = pd.DataFrame({'cur': k_s1, 'prev': prev_key}).groupby('cur', sort=False)['prev'].nunique()
            for v in g[g > 1].index[:2]:
                mem = np.flatnonzero(k_s1 == v)[:3]
                print(f'   [{name}] dangerous collision: ' + '  |  '.join(repr(s1_raw[m]) for m in mem))
        prev_key, prev_eq = k_s1, eq
    out = pd.DataFrame(rows).set_index('representation')

    def role(r):
        if r.pos_equal_pct >= 15 and r.exact_rule_precision_pct >= 50:
            return 'blocking key + feature'
        if r.pos_equal_pct >= 15:
            return 'feature only (ambiguous as a rule)'
        return 'feature only (low coverage)'
    out['recommended_role'] = out.apply(role, axis=1)
    return out


def _jac(a, b):
    A, B = set(a), set(b)
    return len(A & B) / len(A | B) if (A or B) else np.nan


def _c3(s):
    s = f' {s} '
    return [s[i:i + 3] for i in range(len(s) - 2)]


def sim_auc_table(reps, pos_pairs, neg_pairs):
    """AUC of a similarity representation for positives vs random same-country negatives."""
    rows = []
    for name, fn in reps:
        sp_ = np.array([fn(a, b) for a, b in pos_pairs], dtype=float)
        sn_ = np.array([fn(a, b) for a, b in neg_pairs], dtype=float)
        y = np.r_[np.ones(len(sp_)), np.zeros(len(sn_))]
        s = np.nan_to_num(np.r_[sp_, sn_], nan=-1)
        rows.append(dict(similarity=name, AUC=roc_auc_score(y, s), pos_median=np.nanmedian(sp_), neg_median=np.nanmedian(sn_)))
    return pd.DataFrame(rows).set_index('similarity')


def _rf_ratio(a, b):
    return fuzz.ratio(a, b) / 100 if HAS_RF else np.nan


with Timer('name normalisation experiments'):
    P_SLICES = country_slices(P)
    _fitpos = gt_pairs[gt_pairs.fold == 'fit']
    POS = _fitpos.sample(min(CFG['POS_SAMPLE'], len(_fitpos)), random_state=SEED).reset_index(drop=True)
    pos_s1, pos_pi = POS.s1_idx.to_numpy(), POS.pi.to_numpy()
    S1_CK = S1.country_key.astype(object).to_numpy()
    pos_o_ck = P.country_key.take(pos_pi).astype(object).to_numpy()
    pos_o_name, pos_o_addr = P.business_name.take(pos_pi).tolist(), P.business_address.take(pos_pi).tolist()
    S1_RAW_NAME = S1.business_name.astype(object).to_numpy()
    S1_RAW_ADDR = S1.business_address.astype(object).to_numpy()
    _rng = np.random.RandomState(SEED)
    neg_pi = random_same_country(pos_o_ck, P_SLICES, _rng)

    # ---- learned name synonym map (fit-fold positives only) ----
    NAME_MAP.clear()
    _left = [name_core(b) for b in S1.n_basic.take(pos_s1).tolist()]
    _right = [name_core(name_basic(x)) for x in pos_o_name]
    NAME_MAP.update(learn_token_map(_left, _right))
    print(f'learned NAME_MAP ({len(NAME_MAP)} entries):', dict(list(NAME_MAP.items())[:30]))

    NAME_STAGES = [('raw', rep_raw), ('lower + whitespace', rep_lower), ('+ unicode/transliteration', rep_translit),
                   ('+ punct, &→and, TLD, L.L.C. (basic)', rep_nbasic), ('+ legal forms/stopwords/leet/learned map (core)', rep_ncore),
                   ('+ token sort', rep_nsorted), ('squashed (no spaces)', rep_nsquash), ('consonant skeleton (squashed)', rep_nskel)]
    NAME_REP = rep_experiment(NAME_STAGES, S1.business_name, S1_CK, pos_s1, pos_o_name, pos_o_ck, S1_RAW_NAME, pos_o_name)
display(NAME_REP.round(2))

_n = min(30_000, len(POS))
_pos_pairs = list(zip(S1.n_basic.take(pos_s1[:_n]).tolist(), [name_basic(x) for x in pos_o_name[:_n]]))
_neg_pairs = list(zip(S1.n_basic.take(pos_s1[:_n]).tolist(), [name_basic(x) for x in P.business_name.take(neg_pi[:_n]).tolist()]))
NAME_SIM = sim_auc_table([
    ('token Jaccard (core)', lambda a, b: _jac(name_core(a).split(), name_core(b).split())),
    ('char-3gram Jaccard (core)', lambda a, b: _jac(_c3(name_core(a)), _c3(name_core(b)))),
    ('skeleton-token Jaccard', lambda a, b: _jac(name_skel(name_core(a)).split(), name_skel(name_core(b)).split())),
    ('squashed ratio', lambda a, b: _rf_ratio(name_core(a).replace(' ', ''), name_core(b).replace(' ', ''))),
], _pos_pairs, _neg_pairs)
display(NAME_SIM.round(3))

_nr = NAME_REP
finding('Name normalisation trade-off',
        f"exact-equal positives: raw {_nr.pos_equal_pct.iloc[0]:.1f}% → basic {_nr.pos_equal_pct.iloc[3]:.1f}% → core {_nr.pos_equal_pct.iloc[4]:.1f}% "
        f"→ skeleton {_nr.pos_equal_pct.iloc[-1]:.1f}%; S1 collisions raw {_nr.S1_collision_pct.iloc[0]:.1f}% → skeleton {_nr.S1_collision_pct.iloc[-1]:.1f}%",
        'Each aggressive step recovers matches but also merges distinct S1 entities (dangerous for F0.5).',
        'Keep ALL representations: basic/core/squash/skeleton each become separate features; only high-precision ones '
        '(see recommended_role) are used as exact blocking keys; nothing is deleted from the stored raw text.')
finding('Order-insensitive / character-level name similarity',
        '; '.join(f'{i}: AUC {r.AUC:.3f}' for i, r in NAME_SIM.iterrows()),
        'Word re-ordering, typos and concatenated domains defeat exact tokens.',
        'Use token-set/sort ratios, char-3gram TF-IDF cosine and skeleton/squash similarities as pair features; '
        'char/skeleton evidence also drives the dense encoder (Block G).')

## 7. Address normalisation
Address noise includes abbreviations (`Rd/Road`), component re-ordering (`OR, Eugene, 3900 River Road`), native-script state names,
`NULL/N/A` placeholders, landmarks (`near SBI ATM`) and missing components. The same useful/dangerous analysis is repeated, plus numeric-anchor,
postal-code and landmark statistics. A learned address token map (e.g. `ohio→oh`, `telangana→tg`) comes from fit-fold positives only.

In [ ]:
def rep_abasic(s): return addr_basic(s)
def rep_ahand(s): return ' '.join(ADDR_HAND_MAP.get(t, t) for t in addr_basic(s).split())
def rep_acanon(s): return addr_canon(addr_basic(s))
def rep_aset(s): return ' '.join(sorted(set(addr_canon(addr_basic(s)).split())))
def rep_anums(s): return ' '.join(sorted({x.lstrip('0') or '0' for x in _NUM.findall(addr_basic(s))}))


_LANDMARK = {'near', 'opp', 'behind', 'beside', 'besides', 'adjacent', 'nr', 'bh', 'infront', 'next'}


def postal_codes(canon):
    """postal-like numbers: 6-digit tokens (PIN) or 5-digit tokens not followed by a street word (ZIP / code postal)."""
    t = canon.split()
    out = []
    for i, x in enumerate(t):
        if x.isdigit() and (len(x) == 6 or (len(x) == 5 and not (i + 1 < len(t) and t[i + 1].isalpha() and len(t[i + 1]) > 2))):
            out.append(x)
    return out


def _nums(c):
    return {x.lstrip('0') or '0' for x in _NUM.findall(c)}


with Timer('address normalisation experiments'):
    ADDR_MAP.clear()
    _left = [rep_ahand(x) for x in S1.business_address.take(pos_s1).tolist()]
    _right = [rep_ahand(x) for x in pos_o_addr]
    ADDR_MAP.update(learn_token_map(_left, _right))
    print(f'learned ADDR_MAP ({len(ADDR_MAP)} entries):', dict(list(ADDR_MAP.items())[:40]))
    ADDR_STAGES = [('raw', rep_raw), ('lower + whitespace', rep_lower), ('+ unicode/transliteration', rep_translit),
                   ('+ punct, ordinals, placeholders (basic)', rep_abasic), ('+ multilingual street-type map', rep_ahand),
                   ('+ learned token map (canon)', rep_acanon), ('sorted token set', rep_aset), ('numbers only', rep_anums)]
    ADDR_REP = rep_experiment(ADDR_STAGES, S1.business_address, S1_CK, pos_s1, pos_o_addr, pos_o_ck, S1_RAW_ADDR, pos_o_addr)
display(ADDR_REP.round(2))

_neg_addr = P.business_address.take(neg_pi[:_n]).tolist()
_s1c = [addr_canon(addr_basic(x)) for x in S1.business_address.take(pos_s1[:_n]).tolist()]
_poc = [addr_canon(addr_basic(x)) for x in pos_o_addr[:_n]]
_nec = [addr_canon(addr_basic(x)) for x in _neg_addr]


def _anchor_stats(A, B):
    r = Counter()
    for a, b in zip(A, B):
        na, nb = _nums(a), _nums(b)
        pa, pb = set(postal_codes(a)), set(postal_codes(b))
        ka, kb = set(addr_keys(a).split()), set(addr_keys(b).split())
        if na and nb:
            r['both_numbers'] += 1; r['num_shared'] += bool(na & nb); r['num_conflict'] += not (na & nb)
        if pa and pb:
            r['both_postal'] += 1; r['postal_equal'] += bool(pa & pb)
        r['housenum_street_key_shared'] += bool(ka & kb)
        r['set_equal_but_reordered'] += (set(a.split()) == set(b.split()) and a != b and a != '')
        r['n'] += 1
    n = r['n']
    return {'both have numbers %': 100 * r['both_numbers'] / n,
            'numbers overlap | both %': 100 * r['num_shared'] / max(r['both_numbers'], 1),
            'numbers conflict | both %': 100 * r['num_conflict'] / max(r['both_numbers'], 1),
            'both have postal %': 100 * r['both_postal'] / n,
            'postal equal | both %': 100 * r['postal_equal'] / max(r['both_postal'], 1),
            'house#+street key shared %': 100 * r['housenum_street_key_shared'] / n,
            'same tokens, different order %': 100 * r['set_equal_but_reordered'] / n}


ANCHORS = pd.DataFrame({'true pairs': _anchor_stats(_s1c, _poc), 'random same-country pairs': _anchor_stats(_s1c, _nec)})
display(ANCHORS.round(2))
_lm = {k: 100 * np.mean([bool(_LANDMARK & set(addr_basic(x).split())) for x in v]) for k, v in
       [('S1', S1.business_address.take(pos_s1[:_n]).tolist()), ('S2/S3', pos_o_addr[:_n])]}
print('addresses with landmark words (near/opp/behind...) %:', {k: round(v, 2) for k, v in _lm.items()})

ADDR_SIM = sim_auc_table([
    ('token Jaccard (canon)', lambda a, b: _jac(a.split(), b.split())),
    ('char-3gram Jaccard (canon)', lambda a, b: _jac(_c3(a), _c3(b))),
    ('number-set Jaccard', lambda a, b: _jac(_nums(a), _nums(b))),
    ('adjacent-bigram Jaccard', lambda a, b: _jac(list(zip(a.split(), a.split()[1:])), list(zip(b.split(), b.split()[1:])))),
], list(zip(_s1c, _poc)), list(zip(_s1c, _nec)))
display(ADDR_SIM.round(3))

_ar, _an = ADDR_REP, ANCHORS
finding('Address exact matching is weak, token/numeric evidence is strong',
        f"canon exact-equal positives {_ar.pos_equal_pct.loc['+ learned token map (canon)']:.1f}% vs sorted-set {_ar.pos_equal_pct.loc['sorted token set']:.1f}%; "
        f"reordered-only pairs {_an.loc['same tokens, different order %', 'true pairs']:.1f}%; numbers overlap given both have numbers: "
        f"true {_an.loc['numbers overlap | both %', 'true pairs']:.1f}% vs random {_an.loc['numbers overlap | both %', 'random same-country pairs']:.1f}%",
        'Component re-ordering and abbreviations make string equality useless; numbers (house/door/PIN) are stable anchors.',
        '(1) Blocking: TF-IDF address tokens + adjacent bigrams ("3900_river") and house-number+street keys (Block E). '
        '(2) Features: order-free token set/sort ratios, number-set Jaccard/conflict/subset, first-number equality. '
        '(3) Decision: numeric conflict is a strong veto that the GBDT learns.')
finding('Postal codes as a veto',
        f"postal equal given both have one: true {_an.loc['postal equal | both %', 'true pairs']:.1f}% vs random "
        f"{_an.loc['postal equal | both %', 'random same-country pairs']:.1f}% (both have postal in {_an.loc['both have postal %', 'true pairs']:.1f}% of true pairs)",
        'Same-name franchises / homonyms in different towns differ in postal code.',
        'Dedicated postal-code match / conflict / missing features; postal + name-skeleton blocking pass (Block H).')

### 7b. Model-ready representations for the training phase
The frozen normalisation is applied to all train S1 (2.2M) and the pool (10.3M). Next come record-level attributes (lengths, noise flags, name/address
**commonness** in S1 and in the pool) and the hashed sparse matrices that blocking and features share. The entity-level experiment sample `Q`
holds `N_S1_SAMPLE` S1 entities with their fit/tune/hold folds. The whole pool stays in play, so distractor density is realistic.

In [ ]:
REC_COLS = ['n_len', 'n_ntok', 'a_len', 'a_ntok', 'f_domain', 'f_nonascii', 'f_upper', 'f_junk', 'a_missing',
            'nf_s1', 'nf_pool', 'af_s1', 'af_pool']


def _ntok(ser):
    c = ser.str.count(' ').to_numpy(np.float32) + 1
    c[(ser == '').to_numpy(bool)] = 0
    return c


def prepare_frame(df):
    """Frozen normalisation used identically for train and test frames (adds columns in place)."""
    if 'country_key' not in df:
        df['country_key'] = df.country.str.strip().str.lower()
    if 'n_basic' not in df:
        df['n_basic'] = umap(df.business_name, name_basic)
    df['n_core'] = umap(df.n_basic, name_core)
    df['n_squash'] = df.n_core.str.replace(' ', '', regex=False)
    df['n_skel'] = umap(df.n_core, name_skel)
    df['n_alt'] = umap(df.n_basic, name_alt)
    if 'a_basic' not in df:
        df['a_basic'] = umap(df.business_address, addr_basic)
    df['a_canon'] = umap(df.a_basic, addr_canon)
    df['n_len'] = df.n_basic.str.len().to_numpy(np.float32)
    df['n_ntok'] = _ntok(df.n_core)
    df['a_len'] = df.a_canon.str.len().to_numpy(np.float32)
    df['a_ntok'] = _ntok(df.a_canon)
    fl = umap(df.business_name, name_flags, as_str=False).to_numpy(np.int64)
    df['f_domain'] = ((fl & 1) > 0).astype(np.float32)
    df['f_nonascii'] = ((fl & 2) > 0).astype(np.float32)
    df['f_upper'] = ((fl & 4) > 0).astype(np.float32)
    df['f_junk'] = ((fl & 8) > 0).astype(np.float32)
    df['a_missing'] = (df.a_canon == '').to_numpy(bool).astype(np.float32)
    df['num0'] = umap(df.a_canon, first_number, as_str=False).to_numpy(np.int64)
    return df


def add_freq_features(s1df, pdf):
    """Commonness of the name/address among S1 entities and in the pool (log counts)."""
    for col, tag in [('n_core', 'nf'), ('a_canon', 'af')]:
        vs, vp = s1df[col].value_counts(), pdf[col].value_counts()
        for df in (s1df, pdf):
            empty = (df[col] == '').to_numpy(bool)
            a = np.log1p(df[col].map(vs).fillna(0).to_numpy(np.float32)); a[empty] = 0
            b = np.log1p(df[col].map(vp).fillna(0).to_numpy(np.float32)); b[empty] = 0
            df[tag + '_s1'], df[tag + '_pool'] = a, b


# ---- hashed sparse representations (shared by blocking and features) ----
def an_split(s): return s.split()
def an_atok(s): return [t for t in s.split() if not t.isdigit()]
def an_anum(s): return list({x.lstrip('0') or '0' for x in _NUM.findall(s)})
def an_abig(s):
    t = s.split()
    return [a + '_' + b for a, b in zip(t, t[1:])]
def an_apost(s): return postal_codes(s)


_HB, _CB = 2 ** CFG['HASH_BITS'], 2 ** CFG['CHAR_BITS']


def _hv(analyzer):
    return HashingVectorizer(analyzer=analyzer, n_features=_HB, alternate_sign=False, norm=None, binary=True, dtype=np.float32)


HV = {'ntok': _hv(an_split), 'nskel': _hv(an_split), 'atok': _hv(an_atok), 'anum': _hv(an_anum),
      'abig': _hv(an_abig), 'apost': _hv(an_apost),
      'nchar': HashingVectorizer(analyzer='char_wb', ngram_range=(3, 3), n_features=_CB, alternate_sign=False,
                                 norm='l2', dtype=np.float32, lowercase=False)}
MAT_SRC = {'ntok': 'n_core', 'nskel': 'n_skel', 'nchar': 'n_core', 'atok': 'a_canon', 'anum': 'a_canon',
           'abig': 'a_canon', 'apost': 'a_canon'}
TOKEN_SPACES = ['ntok', 'nskel', 'atok', 'anum', 'abig', 'apost']


def _hv_transform(args):
    hv, docs = args
    return hv.transform(docs)


def vectorize(ser, hv, chunk=250_000):
    docs = ser.astype(object).tolist()
    parts = run_parallel(_hv_transform, [(hv, docs[i:i + chunk]) for i in range(0, len(docs), chunk)])
    X = sp.vstack(parts, format='csr') if len(parts) > 1 else parts[0].tocsr()
    X.sort_indices()
    return X


def build_matrices(df, tag):
    with Timer(f'vectorise {tag} ({len(df):,} records)'):
        M = {k: vectorize(df[MAT_SRC[k]], HV[k]) for k in HV}
        log('   nnz/record: ' + ', '.join(f'{k}={M[k].nnz / max(len(df), 1):.1f}' for k in M))
    return M


def fit_idf(Pm):
    """IDF statistics of the pool (label-free; recomputed on the test pool at inference)."""
    N = Pm['ntok'].shape[0]
    R = dict(N=N, DF={}, IDF={}, IDF2={}, RARE={}, W={})
    R['cap'] = cap = max(CFG['DF_CAP_MIN'], CFG['DF_CAP_FRAC'] * N)
    mult = {'ntok': 1.0, 'nskel': CFG['SKEL_WEIGHT'], 'atok': 1.0, 'anum': CFG['NUM_WEIGHT'],
            'abig': CFG['BIGRAM_WEIGHT'], 'apost': 1.0}
    for k in TOKEN_SPACES:
        d = np.bincount(Pm[k].indices, minlength=Pm[k].shape[1]).astype(np.float32)
        R['DF'][k] = d
        R['IDF'][k] = (np.log((N + 1) / (d + 1)) + 1).astype(np.float32)
        R['IDF2'][k] = R['IDF'][k] ** 2
        R['RARE'][k] = (d <= CFG['RARE_DF']).astype(np.float32)
        w = R['IDF'][k] * mult[k]
        w[d > cap] = 0
        R['W'][k] = w.astype(np.float32)
        tot = Pm[k].nnz
        dropped = float(d[d > cap].sum())
        log(f'   {k:6s}: {int((d > 0).sum()):>10,} hashed tokens, {int((d > cap).sum()):>6,} above df-cap {cap:,.0f} '
            f'(= {100 * dropped / max(tot, 1):5.1f}% of token occurrences ignored by retrieval)')
    return R


with Timer('prepare train representations'):
    prepare_frame(S1)
    prepare_frame(P)
    add_freq_features(S1, P)
    Q = S1.iloc[SAMPLE_IDX].reset_index(drop=True)
    Q_FOLD = S1_FOLD[SAMPLE_IDX]
    NQ = len(Q)
    s1_to_q = np.full(NS1, -1, np.int64); s1_to_q[SAMPLE_IDX] = np.arange(NQ)
    _g = gt_pairs[s1_to_q[gt_pairs.s1_idx.to_numpy()] >= 0]
    GT_Q = pd.DataFrame({'qi': s1_to_q[_g.s1_idx.to_numpy()].astype(np.int32), 'pi': _g.pi.to_numpy(np.int32)})
    GT_Q_KEYS = GT_Q.qi.to_numpy(np.int64) * NP + GT_Q.pi.to_numpy(np.int64)      # aligned with GT_Q rows
    GT_Q_FOLD = Q_FOLD[GT_Q.qi.to_numpy()]
    N_TRUE_Q = np.bincount(GT_Q.qi.to_numpy(), minlength=NQ)
    FOLD_Q = {f: np.flatnonzero(Q_FOLD == f) for f in ['fit', 'tune', 'hold']}
    Q_CK = Q.country_key.astype(object).to_numpy()
    del S1_RAW_NAME, S1_RAW_ADDR, S1, gt_pairs, POS   # not needed after this point (commonness stats already computed)
    gc.collect()
    print(f'Q (experiment S1) = {NQ:,}; GT pairs in Q = {len(GT_Q):,}; folds =', {k: len(v) for k, v in FOLD_Q.items()})

Qm = build_matrices(Q, 'Q (train sample)')
Pm = build_matrices(P, 'pool (train)')
with Timer('pool IDF'):
    RET = fit_idf(Pm)
gc.collect()

### Blocking engine (label-free)
The engine offers several independent candidate generators. **None of them receives labels.** Ground truth is only used later, to *measure* recall.

| pass | idea | targets |
|---|---|---|
| A | exact squashed core name (+country) | exact / punctuation / domain variants |
| B | first 8 chars of the name consonant skeleton (+country) | vowel typos, transliteration |
| C | TF-IDF cosine on name tokens + skeleton tokens (top-K, within country) | re-ordering, extra words |
| D | TF-IDF cosine on address tokens + numbers + adjacent bigrams (top-K) | DBA / trade names, renamed businesses |
| E | house-number + street-token key | re-ordered addresses |
| F | combined name/address TF-IDF score (top-K) | typical noisy records |
| G | learned char-n-gram encoder, GPU dense top-K | heavy typos, domains, transliteration |
| H | postal code + first name-skeleton token | same area, noisy name |

Tokens more frequent than the document-frequency cap (`llc`, `road`, `delhi`, …) are ignored by retrieval, following the inverted-index stop-word rule.
Scores are sparse matrix products computed chunk by chunk, one country at a time. The full Cartesian product is never materialised.

In [ ]:
KEY_PASSES = ['A', 'B', 'E', 'H']
PASS_NAMES = {'A': 'A exact squashed name', 'B': 'B name-skeleton prefix', 'C': 'C TF-IDF name', 'D': 'D TF-IDF address',
              'E': 'E house#+street key', 'F': 'F TF-IDF name+address', 'G': 'G learned encoder (GPU)',
              'H': 'H postal + name skeleton'}
PASS_BITS = {p: 1 << i for i, p in enumerate('ABCDEFGH')}
RETR_FIELDS = {'name': ['ntok', 'nskel'], 'addr': ['atok', 'anum', 'abig']}

if HAS_NUMBA:
    @numba.njit
    def _topk_rows(indptr, indices, data, k):
        n = indptr.shape[0] - 1
        tot = 0
        for i in range(n):
            m = indptr[i + 1] - indptr[i]
            tot += m if m < k else k
        out_r = np.empty(tot, np.int32); out_c = np.empty(tot, np.int32)
        out_s = np.empty(tot, np.float32); out_k = np.empty(tot, np.int16)
        p = 0
        for i in range(n):
            s = indptr[i]; e = indptr[i + 1]; m = e - s
            if m == 0:
                continue
            order = np.argsort(-data[s:e])
            kk = m if m < k else k
            for j in range(kk):
                t = s + order[j]
                out_r[p] = i; out_c[p] = indices[t]; out_s[p] = data[t]; out_k[p] = j
                p += 1
        return out_r, out_c, out_s, out_k
else:
    def _topk_rows(indptr, indices, data, k):
        R_, C_, S_, K_ = [], [], [], []
        for i in range(len(indptr) - 1):
            s, e = indptr[i], indptr[i + 1]
            if e == s:
                continue
            d = data[s:e]
            part = np.argpartition(-d, k - 1)[:k] if e - s > k else np.arange(e - s)
            part = part[np.argsort(-d[part], kind='stable')]
            R_.append(np.full(len(part), i, np.int32)); C_.append(indices[s:e][part]); S_.append(d[part])
            K_.append(np.arange(len(part), dtype=np.int16))
        if not R_:
            return np.empty(0, np.int32), np.empty(0, np.int32), np.empty(0, np.float32), np.empty(0, np.int16)
        return (np.concatenate(R_), np.concatenate(C_).astype(np.int32), np.concatenate(S_).astype(np.float32),
                np.concatenate(K_))


def topk_csr(S, k):
    S = S.tocsr()
    return _topk_rows(S.indptr, S.indices, S.data.astype(np.float32, copy=False), k)


def scale_cols(X, w):
    X = sp.csr_matrix((X.data * w[X.indices], X.indices.copy(), X.indptr.copy()), shape=X.shape)
    X.eliminate_zeros()
    return X


def retr_mat(M, rows, fields, W):
    X = sp.hstack([scale_cols(M[f][rows], W[f]) for f in fields], format='csr')
    return sk_normalize(X, norm='l2', copy=False)


def _empty_pass():
    return pd.DataFrame({'qi': np.empty(0, np.int32), 'pi': np.empty(0, np.int32),
                         'score': np.empty(0, np.float32), 'rank': np.empty(0, np.int16)})


def _country_groups(q_rows, q_ck, p_slices, n_pool):
    ck = q_ck[q_rows]
    groups = []
    for c, (s, e) in p_slices.items():
        r = q_rows[ck == c]
        if len(r):
            groups.append((c, r, s, e))
    unknown = ~np.isin(ck, np.array(list(p_slices.keys()), dtype=object))
    if unknown.any():          # a country present in S1 but absent from the pool: search the whole pool
        groups.append(('<country-not-in-pool>', q_rows[unknown], 0, n_pool))
    return groups


def sparse_passes(Qm_, Pm_, q_rows, q_ck, p_slices, R, k, which=('C', 'D', 'F'), chunk=None):
    """Label-free TF-IDF retrieval (passes C/D/F), country by country, chunked sparse products."""
    chunk = chunk or CFG['RETR_CHUNK']
    alpha = CFG['ALPHA_NAME']
    out = {p: [] for p in which}
    need_n, need_a = any(p in which for p in 'CF'), any(p in which for p in 'DF')
    for c, rows_c, ps, pe in _country_groups(np.asarray(q_rows), q_ck, p_slices, Pm_['ntok'].shape[0]):
        t = time.time()
        PN = retr_mat(Pm_, slice(ps, pe), RETR_FIELDS['name'], R['W']).T.tocsr() if need_n else None
        PA = retr_mat(Pm_, slice(ps, pe), RETR_FIELDS['addr'], R['W']).T.tocsr() if need_a else None
        for s in range(0, len(rows_c), chunk):
            qi = rows_c[s:s + chunk]
            SN = (retr_mat(Qm_, qi, RETR_FIELDS['name'], R['W']) @ PN).tocsr() if need_n else None
            SA = (retr_mat(Qm_, qi, RETR_FIELDS['addr'], R['W']) @ PA).tocsr() if need_a else None
            mats = {}
            if 'C' in which:
                mats['C'] = SN
            if 'D' in which:
                mats['D'] = SA
            if 'F' in which:
                mats['F'] = (SN * alpha + SA * (1 - alpha)).tocsr()
            for p, S in mats.items():
                r, cix, sc, rk = topk_csr(S, k)
                out[p].append(pd.DataFrame({'qi': qi[r].astype(np.int32), 'pi': (cix.astype(np.int64) + ps).astype(np.int32),
                                            'score': sc, 'rank': rk}))
        log(f'   sparse retrieval [{c}] {len(rows_c):,} queries x {pe - ps:,} pool: {time.time() - t:.0f}s')
        del PN, PA
        gc.collect()
    return {p: (pd.concat(v, ignore_index=True) if v else _empty_pass()) for p, v in out.items()}


def addr_set_key(canon):
    return ' '.join(sorted(set(canon.split())))


def record_keys(df, kind):
    """(row, uint64 hash) arrays for exact-key blocking. kind: A,B,E,H (+ S,K for hard-negative mining)."""
    ck = df.country_key.astype(object).to_numpy()
    if kind in ('A', 'B', 'K'):
        if kind == 'A':
            k = df.n_squash
            minlen = 3
        else:
            k = df.n_skel.str.replace(' ', '', regex=False)
            minlen = 4
            if kind == 'B':
                k = k.str[:8]
        ln = k.str.len().to_numpy(np.int64)
        rows = np.flatnonzero(ln >= minlen)
        vals = ck[rows] + '|' + k.astype(object).to_numpy()[rows]
    elif kind == 'S':
        k = umap(df.a_canon, addr_set_key).astype(object).to_numpy()
        rows = np.flatnonzero(df.a_ntok.to_numpy() >= 3)
        vals = ck[rows] + '|' + k[rows]
    elif kind in ('E', 'H'):
        if kind == 'E':
            ks = umap(df.a_canon, addr_keys).astype(object).str.split().explode()
        else:
            ks = umap(df.a_canon, lambda_postal_str).astype(object).str.split().explode()
        ks = ks[ks.notna() & (ks != '')]
        rows = ks.index.to_numpy()
        suffix = ks.to_numpy(object)
        if kind == 'H':
            first = df.n_skel.str.split(' ').str[0].str[:3].astype(object).to_numpy()
            suffix = suffix + '|' + first[rows]
        vals = ck[rows] + '|' + suffix
    else:
        raise ValueError(kind)
    h = pd.util.hash_array(np.asarray(vals, dtype=object)) if len(rows) else np.empty(0, np.uint64)
    return np.asarray(rows, np.int64), h


def lambda_postal_str(canon):
    return ' '.join(postal_codes(canon))


KEY_CACHE = {}


def key_pass(kind, Qdf, Pdf, q_rows, tag, max_block=None):
    """Exact-key blocking: join query keys to pool keys; blocks with > max_block pool records are skipped."""
    max_block = max_block or CFG['MAX_BLOCK_POOL']
    if (tag, kind) not in KEY_CACHE:
        KEY_CACHE[(tag, kind)] = (record_keys(Qdf, kind), record_keys(Pdf, kind))
    (qr, qh), (pr, ph) = KEY_CACHE[(tag, kind)]
    sel = np.isin(qr, q_rows)
    qdf = pd.DataFrame({'h': qh[sel], 'qi': qr[sel]})
    pdf_ = pd.DataFrame({'h': ph, 'pi': pr})
    pdf_ = pdf_[pdf_.h.isin(qdf.h.unique())]
    vc = pdf_.h.value_counts()
    ok = vc.index[vc.to_numpy() <= max_block]
    m = qdf[qdf.h.isin(ok)].merge(pdf_, on='h')[['qi', 'pi']].drop_duplicates()
    return pd.DataFrame({'qi': m.qi.to_numpy(np.int32), 'pi': m.pi.to_numpy(np.int32),
                         'score': np.ones(len(m), np.float32), 'rank': np.zeros(len(m), np.int16)})


def dense_pass(Qe, Pe, q_rows, q_ck, p_slices, k, chunk=None):
    """Pass G: exact dense top-K on the GPU (fp16), within country."""
    chunk = chunk or CFG['DENSE_CHUNK']
    R_, C_, S_, K_ = [], [], [], []
    for c, rows_c, ps, pe in _country_groups(np.asarray(q_rows), q_ck, p_slices, len(Pe)):
        t = time.time()
        Pt = torch.from_numpy(Pe[ps:pe]).cuda()
        kk = min(k, pe - ps)
        with torch.no_grad():
            for s in range(0, len(rows_c), chunk):
                qi = rows_c[s:s + chunk]
                v, ix = torch.topk(torch.from_numpy(Qe[qi]).cuda() @ Pt.T, kk, dim=1)
                R_.append(np.repeat(qi, kk).astype(np.int32)); C_.append((ix.cpu().numpy().ravel() + ps).astype(np.int32))
                S_.append(v.float().cpu().numpy().ravel()); K_.append(np.tile(np.arange(kk, dtype=np.int16), len(qi)))
        del Pt
        torch.cuda.empty_cache()
        log(f'   dense retrieval [{c}] {len(rows_c):,} queries x {pe - ps:,} pool: {time.time() - t:.0f}s')
    if not R_:
        return _empty_pass()
    return pd.DataFrame({'qi': np.concatenate(R_), 'pi': np.concatenate(C_), 'score': np.concatenate(S_), 'rank': np.concatenate(K_)})


def union_candidates(pass_dfs, K_sel, n_pool):
    """Union of the selected passes (top-K of ranked passes) → unique (qi, pi) with a bitmask of passes."""
    keys, bits = [], []
    for p, K in K_sel.items():
        d = pass_dfs[p]
        if p not in KEY_PASSES:
            d = d[d['rank'] < K]
        keys.append(d.qi.to_numpy(np.int64) * n_pool + d.pi.to_numpy(np.int64))
        bits.append(np.full(len(d), PASS_BITS[p], np.uint8))
    keys, bits = np.concatenate(keys), np.concatenate(bits)
    uk, inv = np.unique(keys, return_inverse=True)
    ob = np.zeros(len(uk), np.uint8)
    np.bitwise_or.at(ob, inv.ravel(), bits)
    return pd.DataFrame({'qi': (uk // n_pool).astype(np.int32), 'pi': (uk % n_pool).astype(np.int32), 'bits': ob})

## 8. Pair features, and what true matches look like
Features are computed **only for candidate pairs**, in chunks of about 1M pairs. They fall into these groups:
* **Name**: raw/basic/core/squash/skeleton equality; Levenshtein, Jaro-Winkler, ratio, token-sort, token-set and partial ratios (rapidfuzz, C++, multi-threaded);
  squash and skeleton ratios; trade-name (`t/a`, `dba`) ratio; acronym match; token Jaccard/overlap/IDF-weighted Jaccard/cosine; shared **rare** tokens; char-3gram cosine.
* **Address**: raw/basic/canon equality; ratio, token-set, token-sort and partial ratios; token and bigram TF-IDF cosine; number-set Jaccard, conflict and subset; first-number equality;
  **postal** match and conflict; lengths; missingness.
* **Cross-field/context**: source (S2/S3), combined score, name/address *commonness* on both sides, noise flags, which blocking passes found the pair,
  and **within-S1 context** (rank, gap to the best and z-score of key similarities among the S1's candidates; number of candidates).

Country is deliberately **not** a feature, so the model transfers to France.

In [ ]:
def _rf(scorer, a, b, scale):
    if scorer is None:
        return np.full(len(a), np.nan, np.float32)
    if cpdist is not None:
        return (cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32) / scale).astype(np.float32)
    return (np.fromiter((scorer(x, y) for x, y in zip(a, b)), dtype=np.float32, count=len(a)) / scale).astype(np.float32)


def _obj(lst):
    a = np.empty(len(lst), dtype=object)
    a[:] = lst
    return a


def rowdot(A, B, ia, ib):
    return np.asarray(A[ia].multiply(B[ib]).sum(axis=1), dtype=np.float32).ravel()


def _tok_block(f, pre, A, B, qi, pi, R, space, rare=False, sizes=False, seteq=False):
    a, b = A[qi], B[pi]
    inter = a.multiply(b).tocsr()
    na = np.diff(a.indptr).astype(np.float32); nb = np.diff(b.indptr).astype(np.float32)
    sh = np.diff(inter.indptr).astype(np.float32)
    idf, idf2 = R['IDF'][space], R['IDF2'][space]
    ia, ib, ish = a @ idf, b @ idf, inter @ idf
    qa, qb, qsh = a @ idf2, b @ idf2, inter @ idf2
    empty = (na == 0) | (nb == 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        f[pre + '_shared'] = sh
        f[pre + '_jacc'] = np.where(empty, np.nan, sh / np.maximum(na + nb - sh, 1)).astype(np.float32)
        f[pre + '_overlap'] = np.where(empty, np.nan, sh / np.maximum(np.minimum(na, nb), 1)).astype(np.float32)
        f[pre + '_wjacc'] = np.where(empty, np.nan, ish / np.maximum(ia + ib - ish, 1e-6)).astype(np.float32)
        f[pre + '_cos'] = np.where(empty, np.nan, qsh / np.sqrt(np.maximum(qa * qb, 1e-12))).astype(np.float32)
    if rare:
        f[pre + '_rare_shared'] = (inter @ R['RARE'][space]).astype(np.float32)
    if sizes:
        f[pre + '_n1'], f[pre + '_n2'] = na, nb
    if seteq:
        f[pre + '_set_eq'] = ((sh == na) & (sh == nb) & (na > 0)).astype(np.float32)
    return sh, na, nb


def pair_features(qi, pi, Q_, P_, Qm_, Pm_, R, Qe=None, Pe=None):
    """Pairwise features for (Q_[qi], P_[pi]). Identical code path for validation and test."""
    f = {}
    g = lambda df, c, idx: df[c].take(idx).tolist()
    qnr, pnr = g(Q_, 'business_name', qi), g(P_, 'business_name', pi)
    qar, par = g(Q_, 'business_address', qi), g(P_, 'business_address', pi)
    qb, pb = g(Q_, 'n_basic', qi), g(P_, 'n_basic', pi)
    qc, pc = g(Q_, 'n_core', qi), g(P_, 'n_core', pi)
    qs, ps = g(Q_, 'n_squash', qi), g(P_, 'n_squash', pi)
    qk, pk = g(Q_, 'n_skel', qi), g(P_, 'n_skel', pi)
    qalt, palt = g(Q_, 'n_alt', qi), g(P_, 'n_alt', pi)
    qab, pab = g(Q_, 'a_basic', qi), g(P_, 'a_basic', pi)
    qac, pac = g(Q_, 'a_canon', qi), g(P_, 'a_canon', pi)
    Ac1, Ac2 = _obj(qac), _obj(pac)
    a_empty = (Ac1 == '') | (Ac2 == '')

    def eq(a, b, mask=None):
        v = (_obj(a) == _obj(b)).astype(np.float32)
        if mask is not None:
            v[mask] = np.nan
        return v
    f['n_raw_eq'], f['n_basic_eq'], f['n_core_eq'] = eq(qnr, pnr), eq(qb, pb), eq(qc, pc)
    f['n_squash_eq'], f['n_skel_eq'] = eq(qs, ps), eq(qk, pk)
    f['a_raw_eq'], f['a_basic_eq'], f['a_canon_eq'] = eq(qar, par, a_empty), eq(qab, pab, a_empty), eq(qac, pac, a_empty)

    # ---- fuzzy (rapidfuzz C++) ----
    f['n_ratio'] = _rf(fuzz.ratio, qb, pb, 100)
    f['n_lev'] = _rf(RF_Lev.normalized_similarity, qc, pc, 1)
    f['n_jw'] = _rf(RF_JW.normalized_similarity, qc, pc, 1)
    f['n_tsort'] = _rf(fuzz.token_sort_ratio, qc, pc, 100)
    f['n_tset'] = _rf(fuzz.token_set_ratio, qc, pc, 100)
    f['n_partial'] = _rf(fuzz.partial_ratio, qs, ps, 100)
    f['n_sq_ratio'] = _rf(fuzz.ratio, qs, ps, 100)
    f['n_skel_ratio'] = _rf(fuzz.ratio, qk, pk, 100)
    alt1 = _rf(fuzz.ratio, qc, palt, 100); alt1[_obj(palt) == ''] = np.nan
    alt2 = _rf(fuzz.ratio, qalt, pc, 100); alt2[_obj(qalt) == ''] = np.nan
    f['n_alt_ratio'] = np.fmax(alt1, alt2)
    f['n_acronym'] = np.fromiter(((len(s2) >= 2 and acronym(c1) == s2) or (len(s1) >= 2 and acronym(c2) == s1)
                                  for c1, c2, s1, s2 in zip(qc, pc, qs, ps)), dtype=np.float32, count=len(qc))
    for name, sc, scale in [('a_ratio', fuzz.ratio, 100), ('a_tset', fuzz.token_set_ratio, 100),
                            ('a_tsort', fuzz.token_sort_ratio, 100), ('a_partial', fuzz.partial_ratio, 100)]:
        v = _rf(sc, qac, pac, scale); v[a_empty] = np.nan
        f[name] = v

    # ---- sparse token statistics ----
    _tok_block(f, 'n_tok', Qm_['ntok'], Pm_['ntok'], qi, pi, R, 'ntok', rare=True, sizes=True, seteq=True)
    _tok_block(f, 'n_skl', Qm_['nskel'], Pm_['nskel'], qi, pi, R, 'nskel')
    f['n_char_cos'] = rowdot(Qm_['nchar'], Pm_['nchar'], qi, pi)
    _tok_block(f, 'a_tok', Qm_['atok'], Pm_['atok'], qi, pi, R, 'atok', rare=True, seteq=True)
    _tok_block(f, 'a_big', Qm_['abig'], Pm_['abig'], qi, pi, R, 'abig')
    sh, na, nb = _tok_block(f, 'a_num', Qm_['anum'], Pm_['anum'], qi, pi, R, 'anum', sizes=True)
    f['a_num_conflict'] = ((na > 0) & (nb > 0) & (sh == 0)).astype(np.float32)
    f['a_num_subset'] = ((sh == np.minimum(na, nb)) & (np.minimum(na, nb) > 0)).astype(np.float32)
    a_, b_ = Qm_['apost'][qi], Pm_['apost'][pi]
    pa_, pb_ = np.diff(a_.indptr), np.diff(b_.indptr)
    psh = np.diff(a_.multiply(b_).tocsr().indptr)
    f['a_post_eq'] = ((psh > 0)).astype(np.float32)
    f['a_post_conflict'] = ((pa_ > 0) & (pb_ > 0) & (psh == 0)).astype(np.float32)
    f['a_post_missing'] = ((pa_ == 0) | (pb_ == 0)).astype(np.float32)
    q0, p0 = Q_.num0.to_numpy()[qi], P_.num0.to_numpy()[pi]
    f['a_num0_eq'] = np.where((q0 < 0) | (p0 < 0), np.nan, (q0 == p0)).astype(np.float32)

    # ---- cross-field / record level ----
    al = CFG['ALPHA_NAME']
    f['comb_cos'] = (al * np.nan_to_num(f['n_tok_cos']) + (1 - al) * np.nan_to_num(f['a_tok_cos'])).astype(np.float32)
    f['name_x_addr'] = (np.nan_to_num(f['n_tset']) * np.nan_to_num(f['a_tset'], nan=0.5)).astype(np.float32)
    for c in REC_COLS:
        f['q_' + c] = Q_[c].to_numpy()[qi].astype(np.float32)
        f['p_' + c] = P_[c].to_numpy()[pi].astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        f['n_len_ratio'] = (np.minimum(f['q_n_len'], f['p_n_len']) / np.maximum(np.maximum(f['q_n_len'], f['p_n_len']), 1)).astype(np.float32)
        f['a_len_ratio'] = np.where(a_empty, np.nan, np.minimum(f['q_a_len'], f['p_a_len']) /
                                    np.maximum(np.maximum(f['q_a_len'], f['p_a_len']), 1)).astype(np.float32)
    f['both_addr_missing'] = (f['q_a_missing'] * f['p_a_missing']).astype(np.float32)
    f['src_s3'] = (P_.src.to_numpy()[pi] == 3).astype(np.float32)
    f['country_eq'] = (Q_.country_key.take(qi).to_numpy(object) == P_.country_key.take(pi).to_numpy(object)).astype(np.float32)
    if Qe is not None and Pe is not None:
        f['emb_cos'] = (Qe[qi].astype(np.float32) * Pe[pi].astype(np.float32)).sum(1)
    return pd.DataFrame(f)


GROUP_BASE = ['n_tset', 'n_char_cos', 'n_tok_wjacc', 'a_tset', 'a_tok_cos', 'comb_cos', 'name_x_addr', 'emb_cos']


def add_context_features(F, qi, bits, passes):
    """Within-S1 context: which passes found the pair, #candidates, rank / gap-to-best / z-score of key similarities."""
    for p in passes:
        F['blk_' + p] = ((bits & PASS_BITS[p]) > 0).astype(np.float32)
    _, inv, cnt = np.unique(qi, return_inverse=True, return_counts=True)
    F['q_ncand'] = cnt[inv.ravel()].astype(np.float32)
    for c in GROUP_BASE:
        if c not in F:
            continue
        v = pd.Series(F[c].fillna(-1).to_numpy(np.float32))
        grp = v.groupby(qi)
        mx, mu, sd = grp.transform('max').to_numpy(), grp.transform('mean').to_numpy(), grp.transform('std').fillna(0).to_numpy()
        F[c + '_gap'] = (mx - v.to_numpy()).astype(np.float32)
        F[c + '_rank'] = grp.rank(ascending=False, method='min').to_numpy(np.float32)
        F[c + '_z'] = ((v.to_numpy() - mu) / (sd + 1e-3)).astype(np.float32)
    return F


def safe_zone(F):
    """'Safe pruning zone': the name AND the address are both clearly dissimilar."""
    name_sim = np.fmax.reduce([F['n_tok_jacc'].fillna(0).to_numpy(), F['n_skl_jacc'].fillna(0).to_numpy(),
                               F['n_char_cos'].fillna(0).to_numpy()])
    addr_sim = np.fmax(F['a_tok_jacc'].fillna(0).to_numpy(), F['a_num_jacc'].fillna(0).to_numpy())
    return (name_sim < 0.2) & (addr_sim < 0.2)


def group_chunks(qi_sorted, chunk):
    n, bounds, s = len(qi_sorted), [], 0
    while s < n:
        e = min(s + chunk, n)
        if e < n:
            e = int(np.searchsorted(qi_sorted, qi_sorted[e - 1], side='right'))
        bounds.append((s, e))
        s = e
    return bounds


def compute_features(cand, Q_, P_, Qm_, Pm_, R, passes, Qe=None, Pe=None, prune=False, scorer=None, keep_cols=None):
    """Chunked feature computation over qi-sorted candidates.
    scorer=None → returns (features DataFrame, keep mask); otherwise streams: returns (scores, keep mask, kept columns)."""
    qi_all, pi_all, bits_all = cand.qi.to_numpy(), cand.pi.to_numpy(), cand.bits.to_numpy()
    keep = np.ones(len(cand), bool)
    outs, kept = [], []
    bounds = group_chunks(qi_all, CFG['FEAT_CHUNK'])
    for j, (s, e) in enumerate(bounds):
        F = pair_features(qi_all[s:e], pi_all[s:e], Q_, P_, Qm_, Pm_, R, Qe, Pe)
        if prune:
            k = ~safe_zone(F)
            keep[s:e] = k
            F = F[k].reset_index(drop=True)
        add_context_features(F, qi_all[s:e][keep[s:e]], bits_all[s:e][keep[s:e]], passes)
        if scorer is None:
            outs.append(F)
        else:
            outs.append(scorer(F).astype(np.float32))
            if keep_cols:
                kept.append(F[keep_cols].astype(np.float32))
        if j % 10 == 0 or j == len(bounds) - 1:
            log(f'   features chunk {j + 1}/{len(bounds)} ({e:,}/{len(cand):,} pairs)')
    if scorer is None:
        return pd.concat(outs, ignore_index=True), keep
    return np.concatenate(outs), keep, (pd.concat(kept, ignore_index=True) if kept else None)


# ---- what do true matches look like? ----
with Timer('positive-pair features'):
    _fitgt = GT_Q[Q_FOLD[GT_Q.qi.to_numpy()] == 'fit']
    _pp = _fitgt.sample(min(100_000, len(_fitgt)), random_state=SEED).sort_values('qi')
    POSF = pair_features(_pp.qi.to_numpy(), _pp.pi.to_numpy(), Q, P, Qm, Pm, RET)
show_cols = ['n_core_eq', 'n_tset', 'n_jw', 'n_char_cos', 'n_skl_jacc', 'n_sq_ratio', 'a_tset', 'a_tok_cos', 'a_num_jacc',
             'a_num_conflict', 'a_post_eq', 'a_post_conflict', 'p_a_missing', 'p_f_nonascii', 'p_f_domain']
display(POSF[show_cols].describe(percentiles=[.05, .25, .5, .75]).T.round(3))

fig, axs = plt.subplots(2, 4, figsize=(20, 6.5))
for ax, c in zip(axs.ravel(), ['n_tset', 'n_char_cos', 'n_skel_ratio', 'n_jw', 'a_tset', 'a_tok_cos', 'a_num_jacc', 'comb_cos']):
    ax.hist(POSF[c].dropna(), bins=50, color='seagreen'); ax.set_title(f'true pairs: {c}')
plt.tight_layout(); plt.show()

_nj = pd.cut(POSF.n_tok_jacc.fillna(0), [-0.01, .2, .4, .6, .8, 1.0], labels=['0-.2', '.2-.4', '.4-.6', '.6-.8', '.8-1'])
_aj = pd.cut(POSF.a_tok_jacc.fillna(0), [-0.01, .2, .4, .6, .8, 1.0], labels=['0-.2', '.2-.4', '.4-.6', '.6-.8', '.8-1'])
GRID = pd.crosstab(_nj, _aj, normalize='all') * 100
print('true pairs: name-token Jaccard (rows) x address-token Jaccard (cols), % of pairs')
display(GRID.round(2))
_low_name = (POSF.n_tset < 0.5).mean()
finding('Compensation between name and address evidence',
        f'{_low_name:.1%} of true pairs have name token-set ratio < 0.5, of which {(POSF.loc[POSF.n_tset < 0.5, "a_tset"] > 0.8).mean():.1%} '
        f'have address token-set ratio > 0.8; the both-low corner (Jaccard < 0.2 on both) holds {GRID.iloc[0, 0]:.2f}% of true pairs',
        'Trade names/renamed businesses are only recoverable through the address, and address-less records only through the name.',
        'Blocking must include an address-driven pass (D/E) and a name-driven pass (A/B/C); the model gets both families plus '
        'their product; the both-dissimilar corner is a candidate for safe pruning (validated in §12).')
finding('Transliterated pool names',
        f"true pairs with non-ASCII pool name: n_tset median {POSF.loc[POSF.p_f_nonascii == 1, 'n_tset'].median():.2f}, "
        f"skeleton ratio median {POSF.loc[POSF.p_f_nonascii == 1, 'n_skel_ratio'].median():.2f}",
        'Without transliteration these pairs would have zero name similarity.',
        'Keep the Unicode-name transliteration + consonant skeleton; expose the non-ASCII flag so the model can trust skeleton features more for them.')

## 9. Hard-negative construction
Random negatives are trivially separable. Here negatives are mined for a subset of fit-fold S1 entities, using seven label-free strategies.
The question is **which features still separate true matches from each kind of hard negative**.

In [ ]:
with Timer('hard-negative mining'):
    _rng = np.random.RandomState(SEED + 1)
    HQ = np.sort(_rng.choice(FOLD_Q['fit'], min(30_000, len(FOLD_Q['fit'])), replace=False))
    _hp = sparse_passes(Qm, Pm, HQ, Q_CK, P_SLICES, RET, k=10, which=('C', 'D', 'F'))
    _rand = pd.DataFrame({'qi': np.repeat(HQ, 3).astype(np.int32)})
    _rand['pi'] = random_same_country(Q_CK[_rand.qi.to_numpy()], P_SLICES, _rng).astype(np.int32)
    NEG_SOURCES = {
        '1 same-country random': _rand,
        '2 similar name (TF-IDF name top-10)': _hp['C'],
        '3 similar address (TF-IDF addr top-10)': _hp['D'],
        '4 same normalised name': key_pass('A', Q, P, HQ, 'train', max_block=2000),
        '5 same address token-set': key_pass('S', Q, P, HQ, 'train', max_block=2000),
        '6 similar name + similar address': _hp['F'],
        '7 aggressive-normalisation collision': key_pass('K', Q, P, HQ, 'train', max_block=2000),
    }
    _parts = []
    for t, d in NEG_SOURCES.items():
        d = d[['qi', 'pi']].copy()
        d['type'] = t
        _parts.append(d)
    HN = pd.concat(_parts, ignore_index=True)
    HN['key'] = HN.qi.to_numpy(np.int64) * NP + HN.pi.to_numpy(np.int64)
    HN = HN[~np.isin(HN.key.to_numpy(), GT_Q_KEYS)]
    _pos = GT_Q[np.isin(GT_Q.qi.to_numpy(), HQ)].assign(type='0 TRUE MATCH')
    _pos['key'] = _pos.qi.to_numpy(np.int64) * NP + _pos.pi.to_numpy(np.int64)
    HN = pd.concat([HN, _pos], ignore_index=True)
    _u = HN.drop_duplicates('key').sort_values('qi').reset_index(drop=True)
    HF = pair_features(_u.qi.to_numpy(), _u.pi.to_numpy(), Q, P, Qm, Pm, RET)
    HF['key'] = _u.key.to_numpy()
    HNF = HN[['key', 'type']].merge(HF, on='key', how='left')
    _t6 = HNF.type.str.startswith('6')
    HNF = HNF[~_t6 | ((HNF.n_tset >= 0.8) & (HNF.a_tset >= 0.8))]
    _t7 = HNF.type.str.startswith('7')
    HNF = HNF[~_t7 | (HNF.n_basic_eq == 0)]

cnt = HNF.type.value_counts().sort_index()
print(cnt.to_string())
feat_cols = [c for c in HF.columns if c not in ('key', 'country_eq')]
posmask = HNF.type.str.startswith('0')
auc_rows = {}
for t in sorted(HNF.type.unique()):
    if t.startswith('0'):
        continue
    sub = HNF[posmask | (HNF.type == t)]
    if (sub.type == t).sum() < 50:
        continue
    y = sub.type.str.startswith('0').to_numpy()
    auc_rows[t] = {c: roc_auc_score(y, sub[c].fillna(-1).to_numpy()) for c in feat_cols}
AUC_HN = pd.DataFrame(auc_rows)
AUC_HN = AUC_HN.apply(lambda s: np.maximum(s, 1 - s))            # direction-free separability
AUC_HN['worst_case'] = AUC_HN.min(axis=1)
AUC_HN = AUC_HN.sort_values('worst_case', ascending=False)
display(AUC_HN.head(25).round(3))
med = HNF.groupby('type')[['n_tset', 'n_char_cos', 'a_tset', 'a_num_jacc', 'a_num_conflict', 'a_post_conflict', 'p_nf_s1', 'q_nf_pool']].median()
display(med.round(3))

fig, ax = plt.subplots(figsize=(12, 7))
_top = AUC_HN.head(20).drop(columns='worst_case')
im = ax.imshow(_top.to_numpy(), aspect='auto', cmap='viridis', vmin=0.5, vmax=1)
ax.set_yticks(range(len(_top))); ax.set_yticklabels(_top.index)
ax.set_xticks(range(_top.shape[1])); ax.set_xticklabels([c[:28] for c in _top.columns], rotation=35, ha='right')
plt.colorbar(im, label='AUC (true vs negative type)'); ax.set_title('Which features separate true matches from each hard-negative type')
plt.tight_layout(); plt.show()

_hard = AUC_HN.drop(columns='worst_case').drop(columns=[c for c in AUC_HN.columns if c.startswith('1')], errors='ignore')
finding('Hard negatives need different evidence than random ones',
        f"median AUC over features: random negatives {AUC_HN.filter(like='1 same').median().iloc[0]:.3f} vs "
        f"same-normalised-name negatives {AUC_HN.filter(like='4 same').median().iloc[0] if AUC_HN.filter(like='4 same').shape[1] else float('nan'):.3f}; "
        f"best worst-case separators: {', '.join(AUC_HN.index[:6])}",
        'A model trained on random negatives learns "names differ → no match" and has nothing to say about same-name/same-address traps.',
        'Train on blocking candidates (hard negatives by construction); keep the address/number/postal/commonness features that '
        'separate the same-name and same-address traps. E3 quantifies random vs hard negatives.')
del HN, HF, _u, _hp
gc.collect()

## 10. Blocking experiments
Every pass runs on **all** experiment S1 entities against the **full** train pool, restricted to the same country.
Ranked passes (C, D, F, G) keep the top `TOPK_MAX` so that recall@K curves can be drawn and K chosen afterwards.

**Block G** is a learned encoder that uses the GPU. It is a hashed char-3-gram/token *EmbeddingBag* followed by a small MLP, trained with an in-batch contrastive (InfoNCE) loss
on **fit-fold positive pairs only**. It uses no pretrained weights and no external data, so there are no licence or download issues. Its embeddings also feed the
`emb_cos` feature evaluated in §15.

In [ ]:
ENC_FIELDS = ['nchar', 'ntok', 'nskel', 'atok', 'anum', 'abig']


def enc_inputs(M, rows, R):
    B, nf = 2 ** CFG['ENC_BUCKET_BITS'], len(ENC_FIELDS)
    X = None
    for fi, f in enumerate(ENC_FIELDS):
        A = M[f][rows]
        data = A.data if f == 'nchar' else A.data * R['IDF'][f][A.indices]
        A = sp.csr_matrix((data.astype(np.float32), (A.indices % B + fi * B).astype(np.int64), A.indptr.copy()),
                          shape=(A.shape[0], nf * B))
        A.sum_duplicates()
        A = sk_normalize(A, copy=False)
        X = A if X is None else X + A
    X = X.tocsr()
    return (torch.from_numpy(X.indices.astype(np.int64)), torch.from_numpy(X.indptr[:-1].astype(np.int64)),
            torch.from_numpy(X.data.astype(np.float32)))


if torch is not None:
    class HashEncoder(torch.nn.Module):
        def __init__(self, n_buckets, dim):
            super().__init__()
            self.emb = torch.nn.EmbeddingBag(n_buckets, dim, mode='sum', sparse=True)
            torch.nn.init.normal_(self.emb.weight, std=0.05)
            self.mlp = torch.nn.Sequential(torch.nn.LayerNorm(dim), torch.nn.Linear(dim, 2 * dim), torch.nn.GELU(),
                                           torch.nn.Linear(2 * dim, dim))

        def forward(self, idx, off, w):
            x = self.emb(idx, off, per_sample_weights=w)
            return torch.nn.functional.normalize(x + self.mlp(x), dim=-1)


def _dev(t):
    return [x.cuda(non_blocking=True) for x in t]


def train_encoder(Qm_, Pm_, R, pos_qi, pos_pi):
    B, nf, dim = 2 ** CFG['ENC_BUCKET_BITS'], len(ENC_FIELDS), CFG['ENC_DIM']
    model = HashEncoder(nf * B, dim).cuda()
    opt_s = torch.optim.SparseAdam(list(model.emb.parameters()), lr=CFG['ENC_LR'])
    opt_d = torch.optim.AdamW(model.mlp.parameters(), lr=1e-3)
    rng = np.random.RandomState(SEED)
    order = np.argsort(pos_qi, kind='stable')
    sq, spi = pos_qi[order], pos_pi[order]
    uq, start, counts = np.unique(sq, return_index=True, return_counts=True)
    bs, tau = CFG['ENC_BATCH'], CFG['ENC_TAU']
    for ep in range(CFG['ENC_EPOCHS']):
        pick = start + (rng.rand(len(uq)) * counts).astype(np.int64)      # one positive per S1 per epoch → no in-batch false negatives
        a_rows, b_rows = sq[pick], spi[pick]
        perm = rng.permutation(len(a_rows))
        tot, nb = 0.0, 0
        model.train()
        for s in range(0, len(perm) - bs + 1, bs):
            bi = perm[s:s + bs]
            ea = model(*_dev(enc_inputs(Qm_, a_rows[bi], R)))
            eb = model(*_dev(enc_inputs(Pm_, b_rows[bi], R)))
            logits = ea @ eb.T / tau
            lab = torch.arange(len(bi), device='cuda')
            loss = (torch.nn.functional.cross_entropy(logits, lab) + torch.nn.functional.cross_entropy(logits.T, lab)) / 2
            opt_s.zero_grad(); opt_d.zero_grad()
            loss.backward()
            opt_s.step(); opt_d.step()
            tot += loss.item(); nb += 1
        log(f'   encoder epoch {ep + 1}/{CFG["ENC_EPOCHS"]}: InfoNCE loss {tot / max(nb, 1):.4f}')
    model.eval()
    return model


@torch.no_grad() if torch is not None else (lambda f: f)
def embed_all(model, M, R, n, bs=16384):
    out = np.empty((n, CFG['ENC_DIM']), np.float16)
    for s in range(0, n, bs):
        e = min(n, s + bs)
        out[s:e] = model(*_dev(enc_inputs(M, slice(s, e), R))).half().cpu().numpy()
    return out


ENCODER, Qe, Pe = None, None, None
with Timer('blocking passes A/B/E/H (exact keys)'):
    ALL_Q = np.arange(NQ)
    PASS_DFS = {k: key_pass(k, Q, P, ALL_Q, 'train') for k in KEY_PASSES}
with Timer('blocking passes C/D/F (TF-IDF top-K)'):
    PASS_DFS.update(sparse_passes(Qm, Pm, ALL_Q, Q_CK, P_SLICES, RET, k=CFG['TOPK_MAX']))
if CFG['RUN_ENCODER'] and USE_GPU:
    with Timer('train learned encoder (GPU) + pass G'):
        _fp = GT_Q[Q_FOLD[GT_Q.qi.to_numpy()] == 'fit']
        ENCODER = train_encoder(Qm, Pm, RET, _fp.qi.to_numpy(), _fp.pi.to_numpy())
        Qe = embed_all(ENCODER, Qm, RET, NQ)
        Pe = embed_all(ENCODER, Pm, RET, NP)
        PASS_DFS['G'] = dense_pass(Qe, Pe, ALL_Q, Q_CK, P_SLICES, CFG['TOPK_MAX'])
        torch.save(ENCODER.state_dict(), os.path.join(ART_DIR, 'encoder.pt'))
else:
    print('Block G skipped (no GPU or RUN_ENCODER=False).')

POOL_SIZE_C = {c: e - s for c, (s, e) in P_SLICES.items()}
TOTAL_PAIRS = float(sum(POOL_SIZE_C.get(c, NP) for c in Q_CK))


def keys_of(d):
    return np.unique(d.qi.to_numpy(np.int64) * NP + d.pi.to_numpy(np.int64))


def cand_stats(name, keys, K=None):
    hit = np.isin(GT_Q_KEYS, keys)
    per_q = np.bincount((keys // NP).astype(np.int64), minlength=NQ)
    r = dict(strategy=name, K=K, candidate_pairs=len(keys), recall=hit.mean())
    for f in ['fit', 'tune', 'hold']:
        r[f'recall_{f}'] = hit[GT_Q_FOLD == f].mean()
    r.update(avg_per_S1=per_q.mean(), median_per_S1=np.median(per_q), p95_per_S1=np.percentile(per_q, 95),
             max_per_S1=per_q.max(), S1_without_candidates_pct=100 * (per_q == 0).mean(),
             reduction_ratio=1 - len(keys) / TOTAL_PAIRS, missed_true_pairs=int((~hit).sum()))
    return r


rows = [cand_stats(PASS_NAMES[p], keys_of(d if p in KEY_PASSES else d), None if p in KEY_PASSES else CFG['TOPK_MAX'])
        for p, d in PASS_DFS.items()]
BLOCK_TABLE = pd.DataFrame(rows).set_index('strategy')
display(BLOCK_TABLE.round(4))

# recall@K curves and K selection for ranked passes
RANKED = [p for p in PASS_DFS if p not in KEY_PASSES]
RK = {}
for p in RANKED:
    d = PASS_DFS[p]
    RK[p] = [np.isin(GT_Q_KEYS, keys_of(d[d['rank'] < K])).mean() for K in CFG['K_GRID']]
K_SEL = {p: None for p in KEY_PASSES if p in PASS_DFS}
for p in RANKED:
    target = CFG['K_RECALL_KEEP'] * RK[p][-1]
    K_SEL[p] = next(K for K, r in zip(CFG['K_GRID'], RK[p]) if r >= target)
plt.figure(figsize=(8, 4.2))
for p in RANKED:
    plt.plot(CFG['K_GRID'], RK[p], marker='o', label=f'{PASS_NAMES[p]} (K*={K_SEL[p]})')
plt.xlabel('K (top-K per S1)'); plt.ylabel('true-pair recall'); plt.title('recall@K per ranked blocking pass'); plt.legend(); plt.grid(alpha=.3)
plt.show()
print('Selected K per ranked pass:', {p: K_SEL[p] for p in RANKED})

# greedy multi-pass union selected on FIT-fold recall only
_fit_gt = GT_Q_FOLD == 'fit'
_is_fit_q = Q_FOLD == 'fit'
PASS_KEYS = {}
for p, d in PASS_DFS.items():
    dd = d if p in KEY_PASSES else d[d['rank'] < K_SEL[p]]
    PASS_KEYS[p] = keys_of(dd)
_fit_keys = {p: k[_is_fit_q[(k // NP).astype(np.int64)]] for p, k in PASS_KEYS.items()}
SELECTED, cur, cur_rec, hist = [], np.empty(0, np.int64), 0.0, []
while True:
    best = None
    for p in PASS_KEYS:
        if p in SELECTED:
            continue
        u = np.union1d(cur, _fit_keys[p])
        r = np.isin(GT_Q_KEYS[_fit_gt], u).mean()
        if best is None or r > best[1]:
            best = (p, r, u)
    if best is None or best[1] - cur_rec < CFG['GREEDY_MIN_GAIN']:
        break
    SELECTED.append(best[0]); cur, cur_rec = best[2], best[1]
    hist.append(dict(step=len(SELECTED), added=PASS_NAMES[best[0]], fit_recall=cur_rec,
                     fit_pairs_per_S1=len(cur) / max(_is_fit_q.sum(), 1)))
GREEDY = pd.DataFrame(hist)
display(GREEDY.round(4))
print('Selected blocking architecture:', [PASS_NAMES[p] for p in SELECTED])

with Timer('final candidate union'):
    CAND = union_candidates(PASS_DFS, {p: K_SEL[p] for p in SELECTED}, NP)
    CAND_KEYS = CAND.qi.to_numpy(np.int64) * NP + CAND.pi.to_numpy(np.int64)
FINAL_BLOCK = cand_stats('UNION ' + '+'.join(SELECTED), CAND_KEYS)
BLOCK_TABLE = pd.concat([BLOCK_TABLE, pd.DataFrame([FINAL_BLOCK]).set_index('strategy')])
display(BLOCK_TABLE.round(4))

_missed = GT_Q[~np.isin(GT_Q_KEYS, CAND_KEYS)].head(12)
print('Examples of true pairs lost by blocking:')
for qi_, pi_ in zip(_missed.qi, _missed.pi):
    print(f'  S1: {Q.business_name.iloc[qi_]!r:45} | {Q.business_address.iloc[qi_]!r:60}\n'
          f'  S{P.src.iloc[pi_]}: {P.business_name.iloc[pi_]!r:45} | {P.business_address.iloc[pi_]!r}')
_best_single = BLOCK_TABLE.drop(index=BLOCK_TABLE.index[-1]).recall.idxmax()
finding('Multi-pass blocking',
        f"best single pass '{_best_single}' recall {BLOCK_TABLE.loc[_best_single, 'recall']:.4f}; union of {len(SELECTED)} passes "
        f"recall {FINAL_BLOCK['recall']:.4f} with {FINAL_BLOCK['avg_per_S1']:.1f} candidates/S1 (p95 {FINAL_BLOCK['p95_per_S1']:.0f}), "
        f"reduction ratio {FINAL_BLOCK['reduction_ratio']:.6f}; {FINAL_BLOCK['missed_true_pairs']:,} true pairs lost",
        'Recall lost at blocking can never be recovered by the model — it is the ceiling of the whole system.',
        f"Use the greedy-selected union {SELECTED} with K={ {p: K_SEL[p] for p in SELECTED} } for validation AND test.")

## 11. Blocking validation
The blocking functions never received labels. Ground truth was used only for the recall numbers, and pass selection used **fit-fold** recall only.
Below is the final architecture per fold, and the question from §5: do singletons look different once candidates exist?

In [ ]:
rows = []
for f in ['fit', 'tune', 'hold']:
    qs = FOLD_Q[f]
    m = np.isin((CAND_KEYS // NP).astype(np.int64), qs)
    hit = np.isin(GT_Q_KEYS[GT_Q_FOLD == f], CAND_KEYS[m])
    rows.append(dict(fold=f, S1=len(qs), candidate_pairs=int(m.sum()), recall=hit.mean(),
                     reduction_ratio=1 - m.sum() / sum(POOL_SIZE_C.get(c, NP) for c in Q_CK[qs]),
                     avg_candidates_per_S1=m.sum() / len(qs)))
display(pd.DataFrame(rows).set_index('fold').round(5))

_fsc = PASS_DFS['F'].groupby('qi').score.max().reindex(range(NQ)).fillna(0).to_numpy()
_ncand = np.bincount(CAND.qi.to_numpy(), minlength=NQ)
SING = pd.DataFrame({'singleton': N_TRUE_Q == 0, 'n_candidates': _ncand, 'best_TFIDF_score': _fsc})
display(SING.groupby('singleton').describe().T.round(3))
finding('Singletons have weaker best candidates, not fewer candidates',
        f"median best TF-IDF score: singletons {SING[SING.singleton].best_TFIDF_score.median():.3f} vs matched "
        f"{SING[~SING.singleton].best_TFIDF_score.median():.3f}; median #candidates {SING[SING.singleton].n_candidates.median():.0f} vs "
        f"{SING[~SING.singleton].n_candidates.median():.0f}",
        'Blocking always returns look-alikes, so "has candidates" says nothing; the *strength of the best* candidate does.',
        'Singleton control = anchor threshold on the best pair score of each S1 (tuned in §17) + group-context features (gap to best).')
BLOCK_TABLE.to_csv(os.path.join(ART_DIR, 'blocking_table.csv'))

## 12. Local validation protocol
```text
train S1 entities ──hash split──► fit (60%) / tune (20%) / hold (20%)     (entity-level: no S1 or its GT in two folds)
       │                         every S2/S3 record belongs to ≤1 S1 → no pair-label leakage across folds
       ▼
label-free blocking vs FULL pool ─► candidates ─► features ─► model fit on FIT ─► thresholds / rule on TUNE
       ─► S1-level predictions on HOLD ─► macro F0.5 (all hold S1, singletons included)
```
The learned token maps and the encoder use **fit** positives only. Early stopping and every threshold use **tune** only. **Hold** is touched only to report.
The metric follows the official definition: an empty prediction on a singleton scores 1, and any prediction on a singleton scores 0.
Micro (pair-level) precision and recall are reported next to it.

In [ ]:
with Timer('candidate features (all folds)'):
    FE, _ = compute_features(CAND, Q, P, Qm, Pm, RET, SELECTED, Qe, Pe, prune=False)
C_QI, C_PI = CAND.qi.to_numpy(), CAND.pi.to_numpy()
C_Y = np.isin(CAND_KEYS, GT_Q_KEYS).astype(np.int8)
C_SRC3 = FE.src_s3.to_numpy() == 1
print(f'candidate feature table: {FE.shape}, positives {C_Y.mean():.3%}, memory {FE.memory_usage().sum() / 1e9:.2f} GB')

# ---- safe pruning zone (validated) ----
_sz = safe_zone(FE)
_loss = C_Y[_sz].sum() / len(GT_Q)
PRUNE = bool(_loss <= CFG['PRUNE_MAX_RECALL_LOSS'])
finding('Safe pruning zone',
        f'pairs with name AND address similarity < 0.2: {_sz.mean():.1%} of candidates, containing {C_Y[_sz].sum():,} true pairs '
        f'({100 * _loss:.3f}% of all true pairs)',
        'Pruning junk before scoring shrinks candidate_pairs.tsv / inference cost and sharpens group-context features.',
        f"{'APPLY' if PRUNE else 'DO NOT apply'} pruning (limit {100 * CFG['PRUNE_MAX_RECALL_LOSS']:.2f}% recall loss); "
        'the same rule runs inside the test feature stream.')
CTX_PREFIX = ('blk_',)
CTX_SUFFIX = ('_gap', '_rank', '_z')


def drop_context(F):
    return F.drop(columns=[c for c in F.columns if c.startswith(CTX_PREFIX) or c.endswith(CTX_SUFFIX) or c == 'q_ncand'])


if PRUNE:
    keep = ~_sz
    CAND = CAND[keep].reset_index(drop=True)
    CAND_KEYS, C_Y, C_SRC3 = CAND_KEYS[keep], C_Y[keep], C_SRC3[keep]
    FE = drop_context(FE[keep].reset_index(drop=True))
    add_context_features(FE, CAND.qi.to_numpy(), CAND.bits.to_numpy(), SELECTED)
    C_QI, C_PI = CAND.qi.to_numpy(), CAND.pi.to_numpy()
del _sz
gc.collect()
C_FOLD = Q_FOLD[C_QI]
ROWS = {f: np.flatnonzero(C_FOLD == f) for f in ['fit', 'tune', 'hold']}
C_CK = Q_CK[C_QI]


def s1_metrics(sel_qi, sel_y, n_true, q_eval):
    nq = len(n_true)
    n_pred = np.bincount(sel_qi, minlength=nq)[q_eval].astype(np.float64)
    tp = np.bincount(sel_qi, weights=sel_y.astype(np.float64), minlength=nq)[q_eval]
    nt = n_true[q_eval].astype(np.float64)
    with np.errstate(divide='ignore', invalid='ignore'):
        p = np.where(n_pred > 0, tp / np.maximum(n_pred, 1), 1.0)
        r = np.where(nt > 0, tp / np.maximum(nt, 1), 1.0)
        f = np.where((n_pred == 0) & (nt == 0), 1.0, np.where(tp > 0, 1.25 * p * r / (0.25 * p + r), 0.0))
    sing = nt == 0
    return dict(F05=float(f.mean()), P_macro=float(p.mean()), R_macro=float(r.mean()),
                P_micro=float(tp.sum() / max(n_pred.sum(), 1)), R_micro=float(tp.sum() / max(nt.sum(), 1)),
                singleton_acc=float((n_pred[sing] == 0).mean()) if sing.any() else float('nan'),
                false_merge_rate=float(((n_pred - tp) > 0).mean()), avg_pred_per_S1=float(n_pred.mean()), n_S1=int(len(q_eval)))


def best_for_pool(pi, p):
    order = np.lexsort((-p, pi))
    ps = pi[order]
    first = np.r_[True, ps[1:] != ps[:-1]]
    flag = np.zeros(len(p), bool)
    flag[order[first]] = True
    return flag


def decide(qi, p, s3, bp, rk, prm, nq):
    """Decision rule: per-source thresholds, optional exclusivity (argmax S1 per pool record), optional top-n,
    anchor (an S1 gets matches only if its best pair >= t_anchor) and relative margin to the best pair."""
    ok = p >= np.where(s3, prm['t_s3'], prm['t_s2'])
    if prm.get('exclusive'):
        ok &= bp
    if prm.get('top_n'):
        ok &= rk <= prm['top_n']
    if prm.get('t_anchor', 0) > 0 or prm.get('rel', 0) > 0:
        mx = np.zeros(nq, np.float32)
        np.maximum.at(mx, qi[ok], p[ok])
        ok &= mx[qi] >= prm.get('t_anchor', 0)
        if prm.get('rel', 0) > 0:
            ok &= p >= prm['rel'] * mx[qi]
    return ok


class RuleEval:
    """Fast repeated evaluation of decision rules on one fold."""
    def __init__(self, rows, p, fold, min_p=0.02):
        qi, pi, p = C_QI[rows], C_PI[rows], np.asarray(p, np.float32)
        bp = best_for_pool(pi, p)
        rk = pd.Series(p).groupby(qi).rank(ascending=False, method='first').to_numpy()
        k = p >= min_p
        self.qi, self.p, self.y, self.s3, self.bp, self.rk = qi[k], p[k], C_Y[rows][k], C_SRC3[rows][k], bp[k], rk[k]
        self.rows = rows[k]
        self.q_eval = FOLD_Q[fold]

    def select(self, prm):
        return decide(self.qi, self.p, self.s3, self.bp, self.rk, prm, NQ)

    def metrics(self, prm):
        ok = self.select(prm)
        return s1_metrics(self.qi[ok], self.y[ok], N_TRUE_Q, self.q_eval)


T_GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)


def base_prm(t):
    return dict(t_s2=float(t), t_s3=float(t), t_anchor=0.0, exclusive=False, rel=0.0, top_n=0)


def tune_global(ev):
    res = [(t, ev.metrics(base_prm(t))['F05']) for t in T_GRID]
    t, f = max(res, key=lambda x: x[1])
    return base_prm(t), f, pd.DataFrame(res, columns=['t', 'F05'])


def eval_scores(p_tune, p_hold, label):
    """Tune a global threshold on TUNE, report HOLD."""
    prm, f_t, _ = tune_global(RuleEval(ROWS['tune'], p_tune, 'tune'))
    m = RuleEval(ROWS['hold'], p_hold, 'hold').metrics(prm)
    print(f'  {label:45s} t*={prm["t_s2"]:.2f}  tune F0.5={f_t:.4f}  HOLD F0.5={m["F05"]:.4f}  P={m["P_micro"]:.4f}  R={m["R_micro"]:.4f}')
    return prm, m


ORACLE = {f: s1_metrics(C_QI[ROWS[f]][C_Y[ROWS[f]] == 1], np.ones(int(C_Y[ROWS[f]].sum())), N_TRUE_Q, FOLD_Q[f]) for f in ROWS}
print('Oracle (perfect model on these candidates) macro F0.5:', {f: round(v['F05'], 4) for f, v in ORACLE.items()})

## 13. Baselines — an experiment history from trivial to ML
All baselines are evaluated on **hold**. Thresholds, where they exist, are tuned on **tune**.

In [ ]:
def _country_code_arrays():
    codes = {c: i + 1 for i, c in enumerate(sorted(set(P_SLICES) | set(Q_CK)))}
    pc = np.zeros(NP, np.uint64)
    for c, (s, e) in P_SLICES.items():
        pc[s:e] = codes[c]
    qc = np.array([codes[c] for c in Q_CK], dtype=np.uint64)
    return qc, pc


_QCC, _PCC = _country_code_arrays()


def exact_join_baseline(col, with_country, q_rows):
    qv = Q[col].take(q_rows).astype(object).to_numpy()
    qh = pd.util.hash_array(qv)
    ph = pd.util.hash_array(P[col].astype(object).to_numpy())
    if with_country:
        qh = qh + _QCC[q_rows] * np.uint64(0x9E3779B97F4A7C15)
        ph = ph + _PCC * np.uint64(0x9E3779B97F4A7C15)
    q = pd.DataFrame({'h': qh, 'qi': q_rows})[qv != '']
    p = pd.DataFrame({'h': ph, 'pi': np.arange(NP)})
    m = q.merge(p[p.h.isin(q.h)], on='h')
    keys = m.qi.to_numpy(np.int64) * NP + m.pi.to_numpy(np.int64)
    y = np.isin(keys, GT_Q_KEYS)
    return s1_metrics(m.qi.to_numpy(), y, N_TRUE_Q, q_rows)


BASE = {}
with Timer('baselines'):
    hq = FOLD_Q['hold']
    BASE['B1 exact raw name'] = exact_join_baseline('business_name', False, hq)
    BASE['B1 exact normalised (basic) name'] = exact_join_baseline('n_basic', False, hq)
    BASE['B2 exact basic name + country'] = exact_join_baseline('n_basic', True, hq)
    BASE['B2+ exact core name + country'] = exact_join_baseline('n_core', True, hq)
    _b3 = (0.5 * FE.n_tset.fillna(0) + 0.5 * FE.a_tset.fillna(FE.n_tset.fillna(0))).to_numpy()
    _, BASE['B3 weighted name/address token-set (tuned t)'] = eval_scores(_b3[ROWS['tune']], _b3[ROWS['hold']], 'B3 weighted name/address similarity')
    _b4 = FE[['n_ratio', 'n_jw', 'n_char_cos', 'n_tset', 'a_ratio', 'a_tset', 'a_tok_cos']].mean(axis=1, skipna=True).fillna(0).to_numpy()
    _, BASE['B4 mean of fuzzy similarities (tuned t)'] = eval_scores(_b4[ROWS['tune']], _b4[ROWS['hold']], 'B4 fuzzy similarity average')
BASE_TABLE = pd.DataFrame(BASE).T[['F05', 'P_micro', 'R_micro', 'singleton_acc', 'false_merge_rate', 'avg_pred_per_S1']]
display(BASE_TABLE.round(4))
add_experiment('E0', 'Baseline: exact normalised name (no country)', BASE['B1 exact normalised (basic) name'], 'reference point')
add_experiment('E1', 'Name normalisation: core name (legal forms, translit, learned map) + country block',
               BASE['B2+ exact core name + country'],
               'keep' if BASE['B2+ exact core name + country']['F05'] >= BASE['B1 exact normalised (basic) name']['F05'] else 'keep as feature only')

## 14. ML matching models
All models train on **fit-fold blocking candidates**, whose negatives are hard by construction. Early stopping uses the tune fold. Model comparison uses the
F0.5-oriented protocol above (tuned threshold, macro F0.5 on hold), not ROC-AUC.
Following the ensembling rules in the knowledge base, a **linear** model is included for structural diversity next to the GBDTs, and an inter-model correlation check is reported.

In [ ]:
EMB_FEATS = [c for c in FE.columns if c.startswith('emb_cos')] + (['blk_G'] if 'blk_G' in FE else [])
FEATS_ALL = [c for c in FE.columns if c not in ('country_eq',)]
FEATS_BASE = [c for c in FEATS_ALL if c not in EMB_FEATS]
FEATS_NOCTX = [c for c in FEATS_BASE if not (c.startswith(CTX_PREFIX) or c.endswith(CTX_SUFFIX) or c == 'q_ncand')]
FEATS_NAME_ONLY = [c for c in FEATS_BASE if not (c.startswith('a_') or c.startswith('q_a') or c.startswith('p_a') or
                                                 c.startswith('af') or c in ('comb_cos', 'name_x_addr', 'both_addr_missing') or
                                                 c.startswith('comb_cos') or c.startswith('name_x_addr') or c.startswith('a_tset'))]
print(f'features: all={len(FEATS_ALL)} base(classical)={len(FEATS_BASE)} no-context={len(FEATS_NOCTX)} name-only={len(FEATS_NAME_ONLY)}')
_COLIDX = {c: i for i, c in enumerate(FE.columns)}


def X_of(rows, feats):
    return FE.iloc[rows, [_COLIDX[c] for c in feats]].to_numpy(np.float32)


def cap_rows(rows, max_rows, seed=SEED):
    if len(rows) <= max_rows:
        return rows
    qs = np.unique(C_QI[rows])
    keep_q = np.random.RandomState(seed).choice(qs, int(len(qs) * max_rows / len(rows)), replace=False)
    return rows[np.isin(C_QI[rows], keep_q)]


LGB_PARAMS = dict(objective='binary', learning_rate=CFG['LGB_LR'], num_leaves=127, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=255,
                  num_threads=os.cpu_count(), seed=SEED, verbose=-1)


def train_lgb(feats, tr_rows, va_rows, rounds=None, X_tr=None, y_tr=None):
    X_tr = X_of(tr_rows, feats) if X_tr is None else X_tr
    y_tr = C_Y[tr_rows] if y_tr is None else y_tr
    dtr = lgb.Dataset(X_tr, y_tr, feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(X_of(va_rows, feats), C_Y[va_rows], reference=dtr)
    m = lgb.train(LGB_PARAMS, dtr, num_boost_round=rounds or CFG['LGB_ROUNDS'], valid_sets=[dva],
                  callbacks=[lgb.early_stopping(CFG['EARLY_STOP'], verbose=False), lgb.log_evaluation(250)])
    return m


MODELS, VAL_PRED = {}, {}


def predict_model(name, getX):
    m = MODELS[name]
    if m['kind'] == 'ens':
        return np.mean([predict_model(n, getX) for n in m['members']], axis=0)
    X = getX(m['feats'])
    if m['kind'] == 'lgb':
        return m['obj'].predict(X, num_iteration=m['obj'].best_iteration)
    if m['kind'] == 'xgb':
        return m['obj'].predict(xgb.DMatrix(X, missing=np.nan), iteration_range=(0, m['obj'].best_iteration + 1))
    if m['kind'] in ('cat', 'lr'):
        return m['obj'].predict_proba(X)[:, 1]
    raise ValueError(m['kind'])


def register(name, entry):
    MODELS[name] = entry
    VAL_PRED[name] = {f: predict_model(name, lambda feats, f=f: X_of(ROWS[f], feats)).astype(np.float32) for f in ('tune', 'hold')}
    prm, m = eval_scores(VAL_PRED[name]['tune'], VAL_PRED[name]['hold'], name)
    tune_f = tune_global(RuleEval(ROWS['tune'], VAL_PRED[name]['tune'], 'tune'))[1]
    MODELS[name].update(tune_F05=tune_f, hold=m, prm=prm)
    return m


TR_ROWS = cap_rows(ROWS['fit'], CFG['MAX_TRAIN_ROWS'])
print(f'training rows: {len(TR_ROWS):,} (positives {C_Y[TR_ROWS].mean():.2%})')

with Timer('LightGBM (classical features)'):
    _m = train_lgb(FEATS_BASE, TR_ROWS, ROWS['tune'])
    register('lgb', dict(kind='lgb', obj=_m, feats=FEATS_BASE))
    _m.save_model(os.path.join(ART_DIR, 'lgb.txt'))

if CFG['RUN_XGB'] and xgb is not None:
    with Timer('XGBoost'):
        try:
            _Xt, _Xv = X_of(TR_ROWS, FEATS_BASE), X_of(ROWS['tune'], FEATS_BASE)
            _p = dict(objective='binary:logistic', eval_metric='logloss', eta=CFG['LGB_LR'], max_depth=8, subsample=0.8,
                      colsample_bytree=0.8, min_child_weight=5, seed=SEED, tree_method='hist')
            if USE_GPU:
                _p.update(device='cuda') if int(xgb.__version__.split('.')[0]) >= 2 else _p.update(tree_method='gpu_hist')
            _m = xgb.train(_p, xgb.DMatrix(_Xt, C_Y[TR_ROWS], missing=np.nan), CFG['LGB_ROUNDS'],
                           evals=[(xgb.DMatrix(_Xv, C_Y[ROWS['tune']], missing=np.nan), 'tune')],
                           early_stopping_rounds=CFG['EARLY_STOP'], verbose_eval=250)
            del _Xt, _Xv
            register('xgb', dict(kind='xgb', obj=_m, feats=FEATS_BASE))
            _m.save_model(os.path.join(ART_DIR, 'xgb.json'))
        except Exception as e:
            print('XGBoost failed:', e)

if CFG['RUN_CATBOOST'] and catboost is not None:
    with Timer('CatBoost'):
        try:
            from catboost import CatBoostClassifier
            _m = CatBoostClassifier(iterations=CFG['LGB_ROUNDS'], learning_rate=0.08, depth=8, eval_metric='Logloss',
                                    task_type='GPU' if USE_GPU else 'CPU', random_seed=SEED, od_type='Iter',
                                    od_wait=CFG['EARLY_STOP'], verbose=250, thread_count=os.cpu_count())
            _m.fit(X_of(TR_ROWS, FEATS_BASE), C_Y[TR_ROWS], eval_set=(X_of(ROWS['tune'], FEATS_BASE), C_Y[ROWS['tune']]))
            register('catboost', dict(kind='cat', obj=_m, feats=FEATS_BASE))
            _m.save_model(os.path.join(ART_DIR, 'catboost.cbm'))
        except Exception as e:
            print('CatBoost failed:', e)

if CFG['RUN_LR']:
    with Timer('Logistic regression (linear, diversity)'):
        from sklearn.pipeline import make_pipeline
        from sklearn.impute import SimpleImputer
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import LogisticRegression
        _r = cap_rows(TR_ROWS, 1_500_000)
        _m = make_pipeline(SimpleImputer(strategy='constant', fill_value=-1), StandardScaler(),
                           LogisticRegression(max_iter=400, C=1.0))
        _m.fit(X_of(_r, FEATS_BASE), C_Y[_r])
        register('logreg', dict(kind='lr', obj=_m, feats=FEATS_BASE))

_trees = [n for n in ('lgb', 'xgb', 'catboost') if n in MODELS]
if len(_trees) >= 2:
    register('ens_trees', dict(kind='ens', members=_trees))
if 'logreg' in MODELS and len(_trees) >= 1:
    register('ens_trees+lr', dict(kind='ens', members=_trees + ['logreg']))

CORR = pd.DataFrame({n: VAL_PRED[n]['hold'] for n in MODELS if MODELS[n]['kind'] != 'ens'}).corr()
display(CORR.round(4))
MODEL_TABLE = pd.DataFrame({n: dict(tune_F05=MODELS[n]['tune_F05'], hold_F05=MODELS[n]['hold']['F05'], hold_P=MODELS[n]['hold']['P_micro'],
                                    hold_R=MODELS[n]['hold']['R_micro'], t=MODELS[n]['prm']['t_s2'],
                                    hold_AUC=roc_auc_score(C_Y[ROWS['hold']], VAL_PRED[n]['hold']),
                                    hold_logloss=log_loss(C_Y[ROWS['hold']], np.clip(VAL_PRED[n]['hold'], 1e-6, 1 - 1e-6)))
                            for n in MODELS}).T.sort_values('tune_F05', ascending=False)
display(MODEL_TABLE.round(4))
BEST_CLASSICAL = MODEL_TABLE.index[0]

imp = pd.Series(MODELS['lgb']['obj'].feature_importance('gain'), index=FEATS_BASE).sort_values(ascending=False)
imp.head(30)[::-1].plot.barh(figsize=(8, 8), title='LightGBM gain importance (top 30)'); plt.tight_layout(); plt.show()

# reliability of the best classical model on hold
_ph, _yh = VAL_PRED[BEST_CLASSICAL]['hold'], C_Y[ROWS['hold']]
_bins = pd.cut(_ph, np.linspace(0, 1, 11), include_lowest=True)
REL = pd.DataFrame({'pred': _ph, 'y': _yh}).groupby(_bins).agg(mean_pred=('pred', 'mean'), frac_pos=('y', 'mean'), n=('y', 'size'))
display(REL.round(3))
print(f'Brier (hold) = {brier_score_loss(_yh, _ph):.4f}')

# ---- ablations with a quick LightGBM (same protocol) ----
_quick_q = np.sort(np.random.RandomState(SEED).choice(FOLD_Q['fit'], min(CFG['QUICK_S1'], len(FOLD_Q['fit'])), replace=False))
QUICK_ROWS = ROWS['fit'][np.isin(C_QI[ROWS['fit']], _quick_q)]


def quick_eval(feats, label, tr_rows=None, X_tr=None, y_tr=None, rows_tune=None, rows_hold=None):
    m = train_lgb(feats, QUICK_ROWS if tr_rows is None else tr_rows, ROWS['tune'], rounds=CFG['QUICK_ROUNDS'], X_tr=X_tr, y_tr=y_tr)
    pt = m.predict(X_of(ROWS['tune'], feats), num_iteration=m.best_iteration)
    ph = m.predict(X_of(ROWS['hold'], feats), num_iteration=m.best_iteration)
    return eval_scores(pt, ph, label)[1]


with Timer('ablations E2/E3 + country transfer'):
    m_name = quick_eval(FEATS_NAME_ONLY, 'E2a name-only features')
    m_full = quick_eval(FEATS_BASE, 'E2b name + address features')
    add_experiment('E2', f'Address features added to name features (name-only F0.5 {m_name["F05"]:.4f})', m_full,
                   'keep address features' if m_full['F05'] > m_name['F05'] else 'drop address features')
    # E3: random negatives vs hard (blocking) negatives, same S1, same no-context feature set
    _rng = np.random.RandomState(SEED + 7)
    _pos = GT_Q[np.isin(GT_Q.qi.to_numpy(), _quick_q)]
    _neg = pd.DataFrame({'qi': np.repeat(_quick_q, 5).astype(np.int32)})
    _neg['pi'] = random_same_country(Q_CK[_neg.qi.to_numpy()], P_SLICES, _rng).astype(np.int32)
    _rp = pd.concat([_pos[['qi', 'pi']], _neg], ignore_index=True).sort_values('qi').reset_index(drop=True)
    _ry = np.isin(_rp.qi.to_numpy(np.int64) * NP + _rp.pi.to_numpy(np.int64), GT_Q_KEYS).astype(np.int8)
    _RF = pair_features(_rp.qi.to_numpy(), _rp.pi.to_numpy(), Q, P, Qm, Pm, RET, Qe, Pe)
    m_rand = quick_eval(FEATS_NOCTX, 'E3a trained on random negatives', X_tr=_RF[FEATS_NOCTX].to_numpy(np.float32), y_tr=_ry)
    m_hard = quick_eval(FEATS_NOCTX, 'E3b trained on blocking (hard) negatives')
    add_experiment('E3', f'Hard (blocking) negatives instead of random negatives (random-neg F0.5 {m_rand["F05"]:.4f})', m_hard,
                   'train on blocking candidates' if m_hard['F05'] >= m_rand['F05'] else 'check negatives')
    del _RF, _rp
    # country transfer (proxy for the unseen test country): train on one country, evaluate on the other
    TRANSFER = {}
    _cs = [c for c in P_SLICES if (C_CK[ROWS['fit']] == c).sum() > 10000]
    if len(_cs) >= 2:
        c_tr, c_te = _cs[0], _cs[1]
        for label, tr in [(f'train {c_tr} only', QUICK_ROWS[C_CK[QUICK_ROWS] == c_tr]), ('train all countries', QUICK_ROWS)]:
            m = train_lgb(FEATS_BASE, tr, ROWS['tune'], rounds=CFG['QUICK_ROUNDS'])
            rt, rh = ROWS['tune'][C_CK[ROWS['tune']] == c_te], ROWS['hold'][C_CK[ROWS['hold']] == c_te]
            et = RuleEval(rt, m.predict(X_of(rt, FEATS_BASE), num_iteration=m.best_iteration), 'tune')
            et.q_eval = FOLD_Q['tune'][Q_CK[FOLD_Q['tune']] == c_te]
            prm, _, _ = tune_global(et)
            eh = RuleEval(rh, m.predict(X_of(rh, FEATS_BASE), num_iteration=m.best_iteration), 'hold')
            eh.q_eval = FOLD_Q['hold'][Q_CK[FOLD_Q['hold']] == c_te]
            TRANSFER[label] = eh.metrics(prm)['F05']
        print(f'Country transfer — evaluated on {c_te}:', {k: round(v, 4) for k, v in TRANSFER.items()})
        finding('Cross-country generalisation (proxy for France)',
                f"F0.5 on {c_te}: trained on {c_tr} only = {TRANSFER[f'train {c_tr} only']:.4f} vs all countries = {TRANSFER['train all countries']:.4f}",
                'The test contains an unseen country; features must transfer.',
                'Country-agnostic features (no country id, IDF computed per pool, generic legal forms, transliteration) — '
                'a small transfer gap supports using the same model/thresholds for France.')

add_experiment('E5', f'ML model ({BEST_CLASSICAL}, {len(FEATS_BASE)} classical features, global threshold)',
               MODELS[BEST_CLASSICAL]['hold'], f'best classical model = {BEST_CLASSICAL}')
finding('Model family',
        f"tune/hold F0.5: " + ', '.join(f'{n}={MODEL_TABLE.loc[n, "tune_F05"]:.4f}/{MODEL_TABLE.loc[n, "hold_F05"]:.4f}' for n in MODEL_TABLE.index),
        'GBDTs model the non-linear compensation between name/address/number evidence; correlated trees add little when blended.',
        f'Use {BEST_CLASSICAL} as the classical scorer (selected on TUNE F0.5, never on hold).')

## 15. GPU embedding experiment (learned encoder)
Question: does the learned char-n-gram encoder add signal beyond the classical features? The comparison is **classical** versus **classical + `emb_cos` (+ its context + the Block-G flag)**,
with the same LightGBM, same data and same protocol. Embeddings are kept only if hold-out F0.5 improves by at least `EMB_MIN_GAIN`.

In [ ]:
USE_EMB = False
if EMB_FEATS:
    with Timer('embedding ablation'):
        m_q_base = quick_eval(FEATS_BASE, 'E7a classical (quick)')
        m_q_emb = quick_eval(FEATS_ALL, 'E7b classical + embeddings (quick)')
        _gain = m_q_emb['F05'] - m_q_base['F05']
        USE_EMB = _gain >= CFG['EMB_MIN_GAIN']
        _auc_emb = roc_auc_score(C_Y[ROWS['hold']], FE.emb_cos.to_numpy()[ROWS['hold']])
        finding('Learned embeddings',
                f'emb_cos alone AUC {_auc_emb:.4f}; quick-model F0.5 {m_q_base["F05"]:.4f} → {m_q_emb["F05"]:.4f} (gain {_gain:+.4f})',
                'The encoder captures character-level similarity across typos/transliteration jointly for name+address.',
                'KEEP embeddings (feature + Block G)' if USE_EMB else 'DROP embedding features (no meaningful gain); Block G kept only if selected by blocking recall')
        add_experiment('E7', f'Embeddings added (classical quick F0.5 {m_q_base["F05"]:.4f})', m_q_emb, 'keep' if USE_EMB else 'drop')
        if USE_EMB:
            with Timer('LightGBM (classical + embeddings)'):
                _m = train_lgb(FEATS_ALL, TR_ROWS, ROWS['tune'])
                register('lgb_emb', dict(kind='lgb', obj=_m, feats=FEATS_ALL))
                _m.save_model(os.path.join(ART_DIR, 'lgb_emb.txt'))
                _members = ['lgb_emb'] + [n for n in ('xgb', 'catboost') if n in MODELS]
                if len(_members) >= 2:
                    register('ens_emb', dict(kind='ens', members=_members))
else:
    print('No embeddings available (no GPU / encoder disabled) — E7 skipped.')
    add_experiment('E7', 'Embeddings (not available in this run)', MODELS[BEST_CLASSICAL]['hold'], 'skipped')

SEL_TABLE = pd.DataFrame({n: dict(tune_F05=MODELS[n]['tune_F05'], hold_F05=MODELS[n]['hold']['F05']) for n in MODELS}).T.sort_values('tune_F05', ascending=False)
STAGE1_MODEL = SEL_TABLE.index[0]
print('Stage-1 model selected on TUNE F0.5:', STAGE1_MODEL)
display(SEL_TABLE.round(4))

### 15b. Stage-2 reranker: cross-source confirmation
Over 80% of S1 entities have matches in **both** S2 and S3, and the records of one entity agree with each other.
Stage 2 therefore scores each candidate with extra information:
* Its stage-1 probability and its rank and gap within the S1.
* How similar it is to the **best-scoring candidate of the other source**, and to the best *other* candidate of its own source, together with those candidates' probabilities.

Stage-1 scores for the fit fold are **out-of-fold** (2-fold by S1). This keeps stage 2 from learning on overconfident in-sample probabilities.

In [ ]:
def stage2_chunk(qi, pi, s3, p1, P_, Pm_):
    n = len(qi)
    s3 = s3.astype(np.int64)
    df = pd.DataFrame({'qi': qi, 'p1': p1, 'h': (p1 >= 0.5).astype(np.float32), 'hs': (p1 >= 0.5).astype(np.float32)})
    g = df.groupby('qi')
    F = {'p1': p1.astype(np.float32)}
    F['p1_rank'] = g.p1.rank(ascending=False, method='min').to_numpy(np.float32)
    F['p1_qmax'] = g.p1.transform('max').to_numpy(np.float32)
    F['p1_gap'] = F['p1_qmax'] - F['p1']
    F['p1_q_n50'] = g.h.transform('sum').to_numpy(np.float32)
    F['p1_src_n50'] = df.groupby([qi, s3]).hs.transform('sum').to_numpy(np.float32)
    order = np.lexsort((-p1, s3, qi))
    q_o, s_o = qi[order], s3[order]
    first = np.r_[True, (q_o[1:] != q_o[:-1]) | (s_o[1:] != s_o[:-1])]
    start = np.flatnonzero(first)
    size = np.diff(np.r_[start, n])
    best_row = order[start]
    second_row = np.where(size > 1, order[np.minimum(start + 1, n - 1)], -1)
    gid = np.empty(n, np.int64); gid[order] = np.cumsum(first) - 1
    gkey = q_o[start].astype(np.int64) * 2 + s_o[start]
    okey = qi.astype(np.int64) * 2 + (1 - s3)
    pos = np.minimum(np.searchsorted(gkey, okey), len(gkey) - 1)
    other_best = np.where(gkey[pos] == okey, best_row[pos], -1)
    own_best, own_second = best_row[gid], second_row[gid]
    same_other = np.where(own_best == np.arange(n), own_second, own_best)
    for tag, ref in (('xsrc', other_best), ('ssrc', same_other)):
        ok = ref >= 0
        a, b = pi[ok], pi[ref[ok]]
        rp = np.full(n, np.nan, np.float32); rp[ok] = p1[ref[ok]]
        nt = np.full(n, np.nan, np.float32); at = np.full(n, np.nan, np.float32); cc = np.full(n, np.nan, np.float32)
        if ok.any():
            nt[ok] = _rf(fuzz.token_set_ratio, P_.n_core.take(a).tolist(), P_.n_core.take(b).tolist(), 100)
            ac1, ac2 = P_.a_canon.take(a).tolist(), P_.a_canon.take(b).tolist()
            v = _rf(fuzz.token_set_ratio, ac1, ac2, 100)
            v[(_obj(ac1) == '') | (_obj(ac2) == '')] = np.nan
            at[ok] = v
            cc[ok] = rowdot(Pm_['nchar'], Pm_['nchar'], a, b)
        F[tag + '_p1'], F[tag + '_n_tset'], F[tag + '_a_tset'], F[tag + '_char'] = rp, nt, at, cc
        F[tag + '_support'] = (np.nan_to_num(rp) * np.fmax(np.nan_to_num(nt), np.nan_to_num(at))).astype(np.float32)
    return pd.DataFrame(F)


def stage2_frame(qi, pi, s3, p1, P_, Pm_, keep_df):
    parts = [stage2_chunk(qi[s:e], pi[s:e], s3[s:e], p1[s:e], P_, Pm_) for s, e in group_chunks(qi, 2_000_000)]
    S = pd.concat(parts, ignore_index=True)
    return pd.concat([S, keep_df.reset_index(drop=True)], axis=1)


USE_STAGE2 = False
STAGE2 = None
if CFG['RUN_STAGE2'] and lgb is not None:
    with Timer('stage-2 cross-source reranker'):
        s1_name = 'lgb_emb' if (USE_EMB and 'lgb_emb' in MODELS) else 'lgb'
        s1_feats = MODELS[s1_name]['feats']
        _imp = pd.Series(MODELS[s1_name]['obj'].feature_importance('gain'), index=s1_feats).sort_values(ascending=False)
        KEEP_FEATS = list(_imp.index[:CFG['STAGE2_KEEP_FEATS']])
        n_rounds = max(100, int(MODELS[s1_name]['obj'].best_iteration * 1.05))
        # out-of-fold stage-1 predictions on the fit fold (2 folds by S1)
        P1 = np.zeros(len(CAND), np.float32)
        half = (pd.util.hash_array(Q.entity_id.astype(object).to_numpy()) % 2).astype(np.int64)[C_QI]
        for h in (0, 1):
            tr = TR_ROWS[half[TR_ROWS] != h]
            pr = ROWS['fit'][half[ROWS['fit']] == h]
            mh = lgb.train(LGB_PARAMS, lgb.Dataset(X_of(tr, s1_feats), C_Y[tr]), num_boost_round=n_rounds)
            P1[pr] = mh.predict(X_of(pr, s1_feats))
        for f in ('tune', 'hold'):
            P1[ROWS[f]] = VAL_PRED[s1_name][f]
        S2F = stage2_frame(C_QI, C_PI, C_SRC3.astype(np.int64), P1, P, Pm, FE[KEEP_FEATS])
        S2_FEATS = list(S2F.columns)
        _X = lambda rows: S2F.iloc[rows].to_numpy(np.float32)
        m2 = lgb.train(dict(LGB_PARAMS, num_leaves=63), lgb.Dataset(_X(TR_ROWS), C_Y[TR_ROWS], feature_name=S2_FEATS),
                       num_boost_round=CFG['LGB_ROUNDS'], valid_sets=[lgb.Dataset(_X(ROWS['tune']), C_Y[ROWS['tune']])],
                       callbacks=[lgb.early_stopping(CFG['EARLY_STOP'], verbose=False), lgb.log_evaluation(250)])
        p2 = {f: m2.predict(_X(ROWS[f]), num_iteration=m2.best_iteration).astype(np.float32) for f in ('tune', 'hold')}
        prm2, m2_hold = eval_scores(p2['tune'], p2['hold'], 'stage-2 cross-source reranker')
        tune2 = tune_global(RuleEval(ROWS['tune'], p2['tune'], 'tune'))[1]
        gain = tune2 - MODELS[STAGE1_MODEL]['tune_F05']
        USE_STAGE2 = gain >= CFG['STAGE2_MIN_GAIN']
        STAGE2 = dict(stage1=s1_name, keep_feats=KEEP_FEATS, feats=S2_FEATS, model=m2, val=p2, hold=m2_hold)
        m2.save_model(os.path.join(ART_DIR, 'stage2_lgb.txt'))
        imp2 = pd.Series(m2.feature_importance('gain'), index=S2_FEATS).sort_values(ascending=False)
        display(imp2.head(15).round(0).to_frame('gain'))
        finding('Cross-source confirmation (stage 2)',
                f'tune F0.5 {MODELS[STAGE1_MODEL]["tune_F05"]:.4f} → {tune2:.4f} ({gain:+.4f}); hold F0.5 '
                f'{MODELS[STAGE1_MODEL]["hold"]["F05"]:.4f} → {m2_hold["F05"]:.4f}',
                'A candidate that agrees with the confident candidate of the other source is very likely the same entity; '
                'an isolated look-alike is not.',
                'USE stage-2 reranker in the final pipeline' if USE_STAGE2 else 'Stage 2 not kept (gain below threshold)')
        add_experiment('E8', 'Stage-2 cross-source confirmation reranker (OOF stacking)', m2_hold, 'keep' if USE_STAGE2 else 'drop')
        del S2F
        gc.collect()

FINAL_SCORER = 'stage2' if USE_STAGE2 else STAGE1_MODEL
FINAL_P = {f: (STAGE2['val'][f] if USE_STAGE2 else VAL_PRED[STAGE1_MODEL][f]) for f in ('tune', 'hold')}
print('FINAL scorer:', FINAL_SCORER)

## 16. Threshold optimisation
There is no arbitrary `p > 0.5`. Thresholds are searched on the **tune** fold, and the same curve is shown on **hold** to confirm that the operating region is stable.

In [ ]:
EV_T = RuleEval(ROWS['tune'], FINAL_P['tune'], 'tune')
EV_H = RuleEval(ROWS['hold'], FINAL_P['hold'], 'hold')
_curve = []
for t in T_GRID:
    mt, mh = EV_T.metrics(base_prm(t)), EV_H.metrics(base_prm(t))
    _curve.append(dict(threshold=t, tune_F05=mt['F05'], hold_F05=mh['F05'], precision=mh['P_micro'], recall=mh['R_micro'],
                       false_merge_rate=mh['false_merge_rate'], singleton_acc=mh['singleton_acc'], avg_matches_per_S1=mh['avg_pred_per_S1']))
CURVE = pd.DataFrame(_curve)
display(CURVE[np.isclose(CURVE.threshold * 20, np.round(CURVE.threshold * 20))].set_index('threshold').round(4))
_bt = CURVE.threshold[CURVE.tune_F05.idxmax()]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
ax[0].plot(CURVE.threshold, CURVE.tune_F05, label='tune'); ax[0].plot(CURVE.threshold, CURVE.hold_F05, label='hold')
ax[0].axvline(_bt, ls='--', c='k'); ax[0].set_title('threshold vs macro F0.5'); ax[0].legend(); ax[0].grid(alpha=.3)
for c in ['precision', 'recall', 'singleton_acc', 'false_merge_rate']:
    ax[1].plot(CURVE.threshold, CURVE[c], label=c)
ax[1].set_title('hold: pair precision / recall, singleton accuracy, false-merge rate'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.show()
_region = CURVE[CURVE.tune_F05 >= CURVE.tune_F05.max() - 0.002].threshold
print(f'best global threshold (tune) = {_bt:.2f}; operating region within 0.002 F0.5: [{_region.min():.2f}, {_region.max():.2f}]')

# per-source threshold surface
_g = np.round(np.clip(np.arange(_bt - 0.2, _bt + 0.201, 0.04), 0.05, 0.97), 2)
_g = np.unique(_g)
SURF = np.array([[EV_T.metrics(dict(base_prm(_bt), t_s2=float(a), t_s3=float(b)))['F05'] for b in _g] for a in _g])
plt.figure(figsize=(7, 5.5))
plt.imshow(SURF, origin='lower', cmap='magma', extent=[_g[0], _g[-1], _g[0], _g[-1]], aspect='auto')
plt.colorbar(label='tune macro F0.5'); plt.xlabel('threshold S1–S3'); plt.ylabel('threshold S1–S2')
plt.title('Does each source pair need its own threshold?'); plt.show()
_ia, _ib = np.unravel_index(SURF.argmax(), SURF.shape)
finding('Single vs per-source thresholds',
        f'best single t={_bt:.2f} (tune F0.5 {CURVE.tune_F05.max():.4f}); best (t_S2, t_S3)=({_g[_ia]:.2f}, {_g[_ib]:.2f}) '
        f'(tune F0.5 {SURF.max():.4f})',
        'S2 and S3 have different noise profiles (casing, scripts, domains), so their score calibration can differ.',
        'Per-source thresholds are kept only if they beat the single threshold in the staged search of §17.')

## 17. Multi-match decision logic
The output is a *set* per S1, so the rule is searched for in stages on **tune**, each stage accepted only if macro F0.5 improves:
1. global threshold
2. **exclusivity**: each S2/S3 record goes only to its best-scoring S1. The ground truth guarantees at most one owner, and the stage is preferred unless it hurts.
3. per-source thresholds
4. **anchor + expansion**: an S1 receives matches only if its best pair ≥ `t_anchor`; once anchored, further records need only the (lower) expansion threshold. This is the principled singleton gate.
5. relative margin to the best pair

Top-1 selection is evaluated for comparison only, because the problem is one-to-many.

In [ ]:
def tune_full(ev):
    H = []

    def run(prm, stage):
        m = ev.metrics(prm)
        H.append(dict(stage=stage, **prm, F05=m['F05'], P=m['P_micro'], R=m['R_micro'], singleton_acc=m['singleton_acc']))
        return m['F05']
    best, bf = None, -1.0
    for t in T_GRID:
        f = run(base_prm(t), '1 global')
        if f > bf:
            best, bf = base_prm(t), f
    be, bfe = None, -1.0
    for t in T_GRID:
        prm = dict(base_prm(t), exclusive=True)
        f = run(prm, '2 exclusive')
        if f > bfe:
            be, bfe = prm, f
    if bfe >= bf - 1e-4:
        best, bf = be, bfe
    base3 = dict(best)
    grid = np.unique(np.round(np.clip(np.arange(base3['t_s2'] - 0.2, base3['t_s2'] + 0.201, 0.02), 0.05, 0.97), 2))
    for a in grid:
        for b in grid:
            prm = dict(base3, t_s2=float(a), t_s3=float(b))
            f = run(prm, '3 per-source')
            if f > bf + 1e-6:
                best, bf = prm, f
    base4 = dict(best)
    lo = max(base4['t_s2'], base4['t_s3'])
    for ta in np.round(np.arange(lo, 0.99, 0.01), 2):
        for d in (0.0, 0.05, 0.1, 0.15, 0.2, 0.3):
            prm = dict(base4, t_anchor=float(ta), t_s2=float(max(0.05, base4['t_s2'] - d)), t_s3=float(max(0.05, base4['t_s3'] - d)))
            f = run(prm, '4 anchor+expansion')
            if f > bf + 1e-6:
                best, bf = prm, f
    base5 = dict(best)
    for r in (0.3, 0.5, 0.6, 0.7, 0.8, 0.9):
        prm = dict(base5, rel=r)
        f = run(prm, '5 relative margin')
        if f > bf + 1e-6:
            best, bf = prm, f
    return best, bf, pd.DataFrame(H)


with Timer('decision-rule search (tune)'):
    FINAL_PRM, _ftune, RULE_HIST = tune_full(EV_T)
print('FINAL decision rule:', FINAL_PRM, f'(tune F0.5 {_ftune:.4f})')
display(RULE_HIST.groupby('stage').F05.max().round(4).to_frame('best tune F0.5 in stage'))

_stage_best = {s: RULE_HIST[RULE_HIST.stage == s].sort_values('F05').iloc[-1] for s in RULE_HIST.stage.unique()}
_keys = ['t_s2', 't_s3', 't_anchor', 'exclusive', 'rel', 'top_n']
_top1_t = max(T_GRID, key=lambda t: EV_T.metrics(dict(base_prm(t), top_n=1))['F05'])
RULES = {
    'fixed t = 0.5 (naive)': base_prm(0.5),
    'tuned global threshold': {k: _stage_best['1 global'][k] for k in _keys},
    'top-1 only (tuned t)': dict(base_prm(_top1_t), top_n=1),
    'FINAL staged rule': FINAL_PRM,
}
RULE_TABLE = pd.DataFrame({k: EV_H.metrics(v) for k, v in RULES.items()}).T[
    ['F05', 'P_micro', 'R_micro', 'singleton_acc', 'false_merge_rate', 'avg_pred_per_S1']]
display(RULE_TABLE.round(4))
add_experiment('E6', 'Threshold/decision tuning (per-source, exclusivity, anchor+expansion) vs fixed 0.5 '
               f'(0.5 → {RULE_TABLE.loc["fixed t = 0.5 (naive)", "F05"]:.4f})',
               EV_H.metrics(FINAL_PRM), 'use FINAL staged rule')

# behaviour by true multiplicity on hold
_ok = EV_H.select(FINAL_PRM)
_npred = np.bincount(EV_H.qi[_ok], minlength=NQ)[FOLD_Q['hold']]
_ntrue = N_TRUE_Q[FOLD_Q['hold']]
_tp = np.bincount(EV_H.qi[_ok], weights=EV_H.y[_ok], minlength=NQ)[FOLD_Q['hold']]
with np.errstate(divide='ignore', invalid='ignore'):
    _p = np.where(_npred > 0, _tp / np.maximum(_npred, 1), 1.0); _r = np.where(_ntrue > 0, _tp / np.maximum(_ntrue, 1), 1.0)
    _f = np.where((_npred == 0) & (_ntrue == 0), 1.0, np.where(_tp > 0, 1.25 * _p * _r / (0.25 * _p + _r), 0.0))
MULT = pd.DataFrame({'true': np.minimum(_ntrue, 8), 'pred': np.minimum(_npred, 8), 'F05': _f})
display(MULT.groupby('true').agg(S1=('F05', 'size'), mean_F05=('F05', 'mean'), mean_pred=('pred', 'mean')).round(3))
display(pd.crosstab(MULT.true, MULT.pred, rownames=['#true (capped 8)'], colnames=['#predicted (capped 8)']))
finding('Multi-match rule',
        f"hold F0.5: top-1 {RULE_TABLE.loc['top-1 only (tuned t)', 'F05']:.4f} vs tuned global {RULE_TABLE.loc['tuned global threshold', 'F05']:.4f} "
        f"vs final {RULE_TABLE.loc['FINAL staged rule', 'F05']:.4f}; singleton accuracy {RULE_TABLE.loc['FINAL staged rule', 'singleton_acc']:.4f}",
        'Most S1 entities have 2-6 true records; top-1 caps recall. Anchor+expansion separates "is there a match at all" '
        '(singleton control) from "which records belong".',
        f'FINAL rule = {FINAL_PRM}')

## 18. Error analysis on hold
False positives and false negatives under the final rule are assigned to a taxonomy, first matching rule wins.
Missed true pairs that were never candidates are attributed to blocking, and their features are computed separately.
Each category is paired with a concrete fix.

In [ ]:
_hold_rows = EV_H.rows
_sel_rows = _hold_rows[_ok]
FP = FE.iloc[_sel_rows[C_Y[_sel_rows] == 0]].copy()
FP['qi'], FP['pi'] = C_QI[_sel_rows[C_Y[_sel_rows] == 0]], C_PI[_sel_rows[C_Y[_sel_rows] == 0]]
_sel_keys = CAND_KEYS[_sel_rows]
_hold_gt = GT_Q[GT_Q_FOLD == 'hold']
_hold_gt_keys = _hold_gt.qi.to_numpy(np.int64) * NP + _hold_gt.pi.to_numpy(np.int64)
_fn = _hold_gt[~np.isin(_hold_gt_keys, _sel_keys)].sort_values('qi')
_fn_in_cand = np.isin(_fn.qi.to_numpy(np.int64) * NP + _fn.pi.to_numpy(np.int64), CAND_KEYS)
FN = pair_features(_fn.qi.to_numpy(), _fn.pi.to_numpy(), Q, P, Qm, Pm, RET, Qe, Pe)
FN['qi'], FN['pi'], FN['in_candidates'] = _fn.qi.to_numpy(), _fn.pi.to_numpy(), _fn_in_cand


def _v(F, c, fill=0):
    return F[c].fillna(fill).to_numpy()


fp_rules = [
    ('normalisation collision', (_v(FP, 'n_skel_eq') == 1) & (_v(FP, 'n_basic_eq') == 0) & (_v(FP, 'n_tset') < 0.8)),
    ('same name, other branch/location', (_v(FP, 'n_core_eq') == 1) & (_v(FP, 'a_tset', 1) < 0.6)),
    ('common business name', _v(FP, 'p_nf_s1') >= np.log1p(5)),
    ('same address, different business', (_v(FP, 'a_tset') >= 0.9) & (_v(FP, 'n_tset') < 0.6)),
    ('missing address (name-only evidence)', (_v(FP, 'q_a_missing') + _v(FP, 'p_a_missing')) > 0),
    ('only common tokens shared', (_v(FP, 'n_tok_rare_shared') == 0) & (_v(FP, 'n_tok_shared') > 0)),
]
fn_rules = [
    ('blocking miss (never a candidate)', ~FN.in_candidates.to_numpy()),
    ('missing address', _v(FN, 'p_a_missing') > 0),
    ('transliteration', _v(FN, 'p_f_nonascii') > 0),
    ('domain / handle name', _v(FN, 'p_f_domain') > 0),
    ('word reorder', (_v(FN, 'n_tsort') >= 0.95) & (_v(FN, 'n_ratio') < 0.95)),
    ('typo', _v(FN, 'n_ratio') >= 0.8),
    ('abbreviation / acronym', (_v(FN, 'n_acronym') == 1) | (_v(FN, 'n_tset') >= 0.8)),
    ('DBA / trade name', (_v(FN, 'n_tset') < 0.5) & (_v(FN, 'a_tset') >= 0.8)),
]
FIXES = {
    'normalisation collision': 'down-weight skeleton equality when basic names differ; require address agreement',
    'same name, other branch/location': 'stronger number/postal conflict veto; per-S1 exclusivity already applied',
    'common business name': 'raise anchor threshold when name commonness is high (commonness × similarity interaction)',
    'same address, different business': 'name-similarity floor for address-only matches (multi-tenant buildings)',
    'missing address (name-only evidence)': 'separate threshold for pairs without address evidence',
    'only common tokens shared': 'more weight on rare-token overlap; IDF-weighted Jaccard already present',
    'blocking miss (never a candidate)': 'larger K / extra pass (char-n-gram LSH, acronym key) — check recall@K curve',
    'missing address': 'name-only acceptance needs high name similarity + distinctive name',
    'transliteration': 'richer transliteration (schwa deletion, learned char map)',
    'domain / handle name': 'squash/partial-ratio key already; add domain→token segmentation',
    'word reorder': 'token-sort features exist; check threshold for these',
    'typo': 'char n-gram + skeleton similarities; encoder features',
    'abbreviation / acronym': 'acronym blocking key; learned abbreviation map',
    'DBA / trade name': 'address-driven acceptance when numbers/postal agree',
    'severe corruption / other': 'accept loss (precision-weighted metric)',
    'other': 'inspect examples',
}


def taxonomy(F, rules, default):
    lab = np.full(len(F), default, dtype=object)
    done = np.zeros(len(F), bool)
    for name, m in rules:
        m = np.asarray(m, bool) & ~done
        lab[m] = name
        done |= m
    return lab


FP['category'] = taxonomy(FP, fp_rules, 'other')
FN['category'] = taxonomy(FN, fn_rules, 'severe corruption / other')
ERR = pd.concat([
    FP.category.value_counts().rename('count').to_frame().assign(kind='false positive'),
    FN.category.value_counts().rename('count').to_frame().assign(kind='false negative')])
ERR['share_within_kind_%'] = 100 * ERR['count'] / ERR.groupby('kind')['count'].transform('sum')
ERR['proposed_fix'] = [FIXES.get(i, '') for i in ERR.index]
display(ERR.round(2))
for kind, F in [('FALSE POSITIVE', FP), ('FALSE NEGATIVE', FN)]:
    for cat in F.category.value_counts().index[:4]:
        ex = F[F.category == cat].head(2)
        for qi_, pi_ in zip(ex.qi, ex.pi):
            print(f'[{kind} | {cat}]\n   S1 : {Q.business_name.iloc[qi_]!r} | {Q.business_address.iloc[qi_]!r}\n'
                  f'   S{P.src.iloc[pi_]} : {P.business_name.iloc[pi_]!r} | {P.business_address.iloc[pi_]!r}')
ERR.to_csv(os.path.join(ART_DIR, 'error_taxonomy.csv'))

## 19. Frozen final pipeline
Everything selected above is frozen into `FINAL`: normalisation maps, blocking passes and K, pruning, feature list, scorer and decision rule.
The test run in §20 calls **the same functions** (`prepare_frame`, `build_matrices`, `fit_idf`, `key_pass`/`sparse_passes`/`dense_pass`,
`union_candidates`, `compute_features`, `predict_model`/`stage2_chunk`, `decide`), so validation and inference cannot drift apart.

In [ ]:
FINAL = dict(
    passes=SELECTED, K={p: K_SEL[p] for p in SELECTED}, prune=PRUNE, scorer=FINAL_SCORER, stage1=STAGE1_MODEL,
    stage2=(dict(stage1=STAGE2['stage1'], keep_feats=STAGE2['keep_feats'], feats=STAGE2['feats']) if USE_STAGE2 else None),
    use_emb=bool(USE_EMB), decision=FINAL_PRM,
    validation=dict(hold=EV_H.metrics(FINAL_PRM), oracle_hold=ORACLE['hold'], blocking=FINAL_BLOCK),
    cfg=CFG,
)
with open(os.path.join(ART_DIR, 'final_pipeline.json'), 'w') as fh:
    json.dump(FINAL, fh, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
with open(os.path.join(ART_DIR, 'learned_maps.json'), 'w') as fh:
    json.dump(dict(NAME_MAP=NAME_MAP, ADDR_MAP=ADDR_MAP), fh, indent=1)
np.save(os.path.join(ART_DIR, 'hold_final_scores.npy'), FINAL_P['hold'])
print(f"""
FINAL PIPELINE
  normalisation : translit + basic/core/squash/skeleton names ({len(NAME_MAP)} learned name synonyms),
                  basic/canon addresses ({len(ADDR_MAP)} learned address synonyms), postal/number anchors
  blocking      : {' + '.join(PASS_NAMES[p] + (f' (K={K_SEL[p]})' if K_SEL[p] else '') for p in SELECTED)}  | safe pruning: {PRUNE}
  features      : {len(MODELS[STAGE1_MODEL]['feats']) if MODELS[STAGE1_MODEL]['kind'] != 'ens' else len(FEATS_BASE)} stage-1 features
  scorer        : {FINAL_SCORER}  (stage-1 = {STAGE1_MODEL})
  decision      : {FINAL_PRM}
  hold macro F0.5 = {FINAL['validation']['hold']['F05']:.4f}  (oracle ceiling given candidates {ORACLE['hold']['F05']:.4f})
""")

## 20. Test inference
Training-phase memory is released first. The test files are then reloaded and pushed through the frozen pipeline.
The IDF statistics are **recomputed on the test pool**. This is label-free and matches the training procedure, which computed them on the train pool.
Candidates are generated for **every** test S1, France included. Features and scores are streamed in chunks, and the exclusivity rule runs across all test S1.

In [ ]:
for _name in ['FE', 'Qm', 'Pm', 'PASS_DFS', 'CAND', 'CAND_KEYS', 'POSF', 'HNF', 'Qe', 'Pe', 'FP', 'FN', 'SG', 'sg',
              'S1', 'P', 'Q', 'gt_pairs', 'PASS_KEYS', '_fit_keys', 'cur', 'EV_T', 'EV_H']:
    if _name in globals():
        del globals()[_name]
KEY_CACHE.clear()
gc.collect()
log('training-phase memory released')

with Timer('load + prepare test'):
    T1 = load_tsv(PATHS['test_s1'])
    TP = build_pool(load_tsv(PATHS['test_s2']), load_tsv(PATHS['test_s3']))
    T1['country_key'] = T1.country.str.strip().str.lower()
    prepare_frame(T1)
    prepare_frame(TP)
    add_freq_features(T1, TP)
    NQT, NPT = len(T1), len(TP)
    T_SLICES = country_slices(TP)
    T_CK = T1.country_key.astype(object).to_numpy()
    print(f'test S1={NQT:,}  test pool={NPT:,}  countries in pool: { {c: e - s for c, (s, e) in T_SLICES.items()} }')
    print('test S1 countries:', pd.Series(T_CK).value_counts().to_dict())

Tm = build_matrices(T1, 'test S1')
TPm = build_matrices(TP, 'test pool')
with Timer('test pool IDF'):
    RT = fit_idf(TPm)
TQe = TPe = None
if ENCODER is not None and ('G' in SELECTED or USE_EMB or (USE_STAGE2 and FINAL['stage2'] and 'emb' in FINAL['stage2']['stage1'])):
    with Timer('test embeddings (GPU)'):
        TQe = embed_all(ENCODER, Tm, RT, NQT)
        TPe = embed_all(ENCODER, TPm, RT, NPT)

with Timer('test candidate generation'):
    T_ALL = np.arange(NQT)
    TPASS = {}
    for p in SELECTED:
        if p in KEY_PASSES:
            TPASS[p] = key_pass(p, T1, TP, T_ALL, 'test')
    _ranked = tuple(p for p in SELECTED if p in ('C', 'D', 'F'))
    if _ranked:
        TPASS.update(sparse_passes(Tm, TPm, T_ALL, T_CK, T_SLICES, RT, k=max(K_SEL[p] for p in _ranked), which=_ranked))
    if 'G' in SELECTED:
        TPASS['G'] = dense_pass(TQe, TPe, T_ALL, T_CK, T_SLICES, K_SEL['G'])
    TCAND = union_candidates(TPASS, {p: K_SEL[p] for p in SELECTED}, NPT)
    del TPASS
    gc.collect()
    print(f'test candidates (before pruning): {len(TCAND):,} ({len(TCAND) / NQT:.1f} per S1)')

with Timer('test features + scoring (streamed)'):
    if USE_STAGE2:
        _s1 = FINAL['stage2']['stage1']
        _p1, _keep, _KEEPDF = compute_features(
            TCAND, T1, TP, Tm, TPm, RT, SELECTED, TQe, TPe, prune=PRUNE,
            scorer=lambda F: predict_model(_s1, lambda feats: F[feats].to_numpy(np.float32)),
            keep_cols=FINAL['stage2']['keep_feats'])
        TCAND = TCAND[_keep].reset_index(drop=True)
        _qi, _pi = TCAND.qi.to_numpy(), TCAND.pi.to_numpy()
        _s3 = (TP.src.to_numpy()[_pi] == 3).astype(np.int64)
        PT = np.empty(len(TCAND), np.float32)
        _m2 = STAGE2['model']
        for s, e in group_chunks(_qi, 2_000_000):
            S = pd.concat([stage2_chunk(_qi[s:e], _pi[s:e], _s3[s:e], _p1[s:e], TP, TPm),
                           _KEEPDF.iloc[s:e].reset_index(drop=True)], axis=1)
            PT[s:e] = _m2.predict(S[FINAL['stage2']['feats']].to_numpy(np.float32), num_iteration=_m2.best_iteration)
        del _KEEPDF, _p1
    else:
        PT, _keep, _ = compute_features(
            TCAND, T1, TP, Tm, TPm, RT, SELECTED, TQe, TPe, prune=PRUNE,
            scorer=lambda F: predict_model(FINAL_SCORER, lambda feats: F[feats].to_numpy(np.float32)))
        TCAND = TCAND[_keep].reset_index(drop=True)
    gc.collect()

with Timer('test decision rule'):
    _qi, _pi = TCAND.qi.to_numpy(), TCAND.pi.to_numpy()
    _s3b = TP.src.to_numpy()[_pi] == 3
    _bp = best_for_pool(_pi, PT)
    _rk = (pd.Series(PT).groupby(_qi).rank(ascending=False, method='first').to_numpy()
           if FINAL_PRM.get('top_n') else np.ones(len(PT)))
    TSEL = decide(_qi, PT, _s3b, _bp, _rk, FINAL_PRM, NQT)
    print(f'final test matches: {TSEL.sum():,} pairs; S1 with >=1 match: {len(np.unique(_qi[TSEL])):,} / {NQT:,}')
    _cc = pd.DataFrame({'ck': T_CK[_qi[TSEL]]}).ck.value_counts()
    print('matched pairs per country:', _cc.to_dict())


def write_id_lists(path, header, q_ids, qi_sorted, pi_sorted, pool_ids):
    """One row per S1 (in test_source1 order), comma-joined S2/S3 ids, empty when none. qi must be sorted."""
    n = len(q_ids)
    starts = np.searchsorted(qi_sorted, np.arange(n), side='left')
    ends = np.searchsorted(qi_sorted, np.arange(n), side='right')
    with open(path, 'w', encoding='utf-8', newline='\n') as fh:
        fh.write('\t'.join(header) + '\n')
        for b in range(0, n, 200_000):
            lines = [q_ids[q] + '\t' + ','.join(pool_ids[pi_sorted[starts[q]:ends[q]]]) for q in range(b, min(n, b + 200_000))]
            fh.write('\n'.join(lines) + '\n')


with Timer('write submission files'):
    OUT_MATCH = os.path.join(CFG['WORK_DIR'], 'matching_results.tsv')
    OUT_CAND = os.path.join(CFG['WORK_DIR'], 'candidate_pairs.tsv')
    _qids = T1.entity_id.astype(object).to_numpy()
    _pids = TP.entity_id.astype(object).to_numpy()
    write_id_lists(OUT_CAND, ['source1_entity_id', 'candidate_entity_ids'], _qids, _qi, _pi, _pids)
    write_id_lists(OUT_MATCH, ['source1_entity_id', 'matched_entity_ids'], _qids, _qi[TSEL], _pi[TSEL], _pids)
    os.makedirs(os.path.join(CFG['WORK_DIR'], 'output'), exist_ok=True)
    for f in (OUT_MATCH, OUT_CAND):          # also expose the package layout output/ (hard link, no extra disk)
        dst = os.path.join(CFG['WORK_DIR'], 'output', os.path.basename(f))
        try:
            if os.path.exists(dst):
                os.remove(dst)
            os.link(f, dst)
        except OSError:
            pass
for f in (OUT_MATCH, OUT_CAND):
    print(f'{f}: exists={os.path.exists(f)} size={os.path.getsize(f) / 1e6:,.1f} MB')

## 21. Submission validation
There are two independent checks:
1. An in-notebook validator that re-reads both files and checks every rule: header, one row per test S1, only S2/S3 ids that exist in the test pool,
   no duplicates, matches ⊆ candidates, and no S1 ids.
2. The **official competition validator** (`student_resource/utils/validate_submission.py`, stdlib only), embedded verbatim and run with `--check-ids`.
   Any validator script found under `/kaggle/input` is run as well.

In [ ]:
def validate_submission(match_path, cand_path, s1_ids, pool_ids):
    """Memory-light re-read of both files: the matching file is held in memory (small), the candidate file is streamed."""
    s1_set, pool_set = set(s1_ids), set(pool_ids)
    report = {}
    M, dup_m, n_m = {}, 0, 0
    bad_prefix = dup_in_list = unknown = 0
    with open(match_path, encoding='utf-8') as fh:
        report['matching header ok'] = fh.readline().rstrip('\n').split('\t') == ['source1_entity_id', 'matched_entity_ids']
        for line in fh:
            if not line.strip():
                continue
            s1, _, rest = line.rstrip('\n').partition('\t')
            ids = rest.split(',') if rest else []
            n_m += 1
            dup_m += s1 in M
            M[s1] = ids
            dup_in_list += len(ids) != len(set(ids))
            for i in ids:
                if not (i.startswith('S2-') or i.startswith('S3-')):
                    bad_prefix += 1
                elif i not in pool_set:
                    unknown += 1
    seen_c, dup_c, n_c, cand_bad, cand_dup, not_subset, with_c, tot_c = set(), 0, 0, 0, 0, 0, 0, 0
    with open(cand_path, encoding='utf-8') as fh:
        report['candidate header ok'] = fh.readline().rstrip('\n').split('\t') == ['source1_entity_id', 'candidate_entity_ids']
        for line in fh:
            if not line.strip():
                continue
            s1, _, rest = line.rstrip('\n').partition('\t')
            ids = rest.split(',') if rest else []
            n_c += 1
            dup_c += s1 in seen_c
            seen_c.add(s1)
            cs = set(ids)
            cand_dup += len(ids) != len(cs)
            cand_bad += sum(1 for i in ids if i not in pool_set or not (i.startswith('S2-') or i.startswith('S3-')))
            not_subset += bool(set(M.get(s1, [])) - cs)
            with_c += bool(ids)
            tot_c += len(ids)
    report['no duplicate S1 rows'] = dup_m == 0 and dup_c == 0
    report['every test S1 exactly once (matching)'] = set(M) == s1_set and n_m == len(s1_set)
    report['every test S1 exactly once (candidates)'] = seen_c == s1_set and n_c == len(s1_set)
    report['only S2/S3 ids in matches'] = bad_prefix == 0
    report['no duplicate ids within a list'] = dup_in_list == 0 and cand_dup == 0
    report['all matched ids exist in test pool'] = unknown == 0
    report['all candidate ids exist in test pool (S2/S3 only)'] = cand_bad == 0
    report['matches subset of candidates'] = not_subset == 0
    stats = dict(n_S1=len(s1_set), with_candidates=with_c, with_matches=sum(1 for v in M.values() if v),
                 singletons=sum(1 for v in M.values() if not v), total_candidate_pairs=tot_c,
                 total_matches=sum(len(v) for v in M.values()))
    return all(report.values()), report, stats


OFFICIAL_VALIDATOR_SRC = r'''#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

for _name in ['Tm', 'TPm', 'TQe', 'TPe', 'TCAND', 'PT', '_bp', '_rk', '_s3b']:
    if _name in globals():
        del globals()[_name]
gc.collect()
with Timer('submission validation'):
    VALID_OK, VALID_REPORT, SUB_STATS = validate_submission(OUT_MATCH, OUT_CAND, T1.entity_id.astype(object).tolist(),
                                                            TP.entity_id.astype(object).tolist())
    display(pd.Series(VALID_REPORT, name='check passed').to_frame())
    _vpath = os.path.join(CFG['WORK_DIR'], 'validate_submission.py')
    with open(_vpath, 'w', encoding='utf-8') as fh:
        fh.write(OFFICIAL_VALIDATOR_SRC)
    OFFICIAL_RC = None
    _tdir = os.path.dirname(PATHS['test_s1'])
    # two memory-safe runs (as the validator docstring recommends): id-existence on the scored file, then all rules incl. candidates
    _runs = [['--matching', OUT_MATCH, '--candidate', os.path.join(CFG['WORK_DIR'], '__none__.tsv'), '--test-dir', _tdir, '--check-ids'],
             ['--matching', OUT_MATCH, '--candidate', OUT_CAND, '--test-dir', _tdir]]
    for vp in [_vpath] + [v for v in VALIDATORS if v.endswith('validate_submission.py')]:
        for args in _runs:
            try:
                r = subprocess.run([sys.executable, vp] + args, capture_output=True, text=True, timeout=3600)
                flags = ' '.join(a for a in args if a in ('--check-ids',)) or '(with candidates)'
                print(f'--- official validator {vp} {flags}: exit code {r.returncode}')
                print(r.stdout[-3000:], r.stderr[-2000:])
                OFFICIAL_RC = r.returncode if OFFICIAL_RC is None else max(OFFICIAL_RC, r.returncode)
            except Exception as e:
                print('validator could not run:', e)
SUBMISSION_PASS = VALID_OK and (OFFICIAL_RC in (None, 0))
print(f"""
Submission validation: {'PASS' if SUBMISSION_PASS else 'FAIL'}
Number of S1 entities        : {SUB_STATS['n_S1']:,}
Number with candidates       : {SUB_STATS['with_candidates']:,}
Number with predicted matches: {SUB_STATS['with_matches']:,}
Number predicted as singleton: {SUB_STATS['singletons']:,}
Total candidate pairs        : {SUB_STATS['total_candidate_pairs']:,}
Total final matches          : {SUB_STATS['total_matches']:,}
""")

## 22. Compute and memory report

In [ ]:
TIME_TABLE = pd.DataFrame(TIMINGS)
display(TIME_TABLE)
print(f'total runtime {(time.time() - T0) / 60:.1f} min; current RSS {rss_gb():.1f} GB; GPU used: {USE_GPU}')
TIME_TABLE.to_csv(os.path.join(ART_DIR, 'timings.csv'), index=False)

## 23. Final competition report
This section prints the report and experiment table, and saves all findings. It also writes a pre-filled methodology document (`Documentation_template_filled.md`) for the submission package.

In [ ]:
EXP_TABLE = pd.DataFrame(EXPERIMENTS)
EXP_TABLE['_o'] = EXP_TABLE.Experiment.str[1:].astype(int)
EXP_TABLE = EXP_TABLE.sort_values('_o').drop(columns='_o').reset_index(drop=True)
EXP_TABLE.to_csv(os.path.join(ART_DIR, 'experiments.csv'), index=False)
pd.DataFrame(FINDINGS).to_csv(os.path.join(ART_DIR, 'findings.csv'), index=False)
_hold = FINAL['validation']['hold']
try:
    EXP_MD = EXP_TABLE.to_markdown(index=False)
except Exception:
    EXP_MD = EXP_TABLE.to_string(index=False)
REPORT = f"""
DATASET
-------
TRAIN S1: {NS1:,}
TRAIN S2+S3 (pool): {NP:,}
TEST  S1 / pool: {NQT:,} / {NPT:,}
SINGLETON RATE (train): {(N_TRUE_ALL == 0).mean():.2%}

BLOCKING
--------
FINAL BLOCKING RECALL (validation sample): {FINAL_BLOCK['recall']:.4f}  (hold {FINAL_BLOCK['recall_hold']:.4f})
AVERAGE CANDIDATES/S1: {FINAL_BLOCK['avg_per_S1']:.1f}  (test: {SUB_STATS['total_candidate_pairs'] / max(NQT, 1):.1f} after pruning={PRUNE})
CANDIDATE REDUCTION: {FINAL_BLOCK['reduction_ratio']:.6f}

MODEL
-----
MODEL: {FINAL_SCORER} (stage-1: {STAGE1_MODEL})
FEATURE COUNT: {len(FEATS_ALL if USE_EMB else FEATS_BASE)} stage-1{f" + {len(STAGE2['feats'])} stage-2" if USE_STAGE2 else ''}

VALIDATION (hold fold, macro F0.5 per S1)
----------
PRECISION (pair-level): {_hold['P_micro']:.4f}
RECALL    (pair-level): {_hold['R_micro']:.4f}
F0.5: {_hold['F05']:.4f}   (oracle ceiling {ORACLE['hold']['F05']:.4f}; singleton acc {_hold['singleton_acc']:.4f})

FINAL PIPELINE
--------------
NORMALISATION: Unicode-name transliteration, basic/core/squash/skeleton names, legal-form & learned synonym maps, canonical addresses, number/postal anchors
BLOCKING: {' + '.join(PASS_NAMES[p] for p in SELECTED)} (within country) {'+ safe pruning' if PRUNE else ''}
FEATURES: fuzzy (rapidfuzz), TF-IDF/IDF-weighted token & char statistics, numeric/postal agreement, commonness, noise flags, within-S1 context{', learned embeddings' if USE_EMB else ''}
MODEL: {FINAL_SCORER}
THRESHOLD: t_S2={FINAL_PRM['t_s2']:.2f}, t_S3={FINAL_PRM['t_s3']:.2f}, anchor={FINAL_PRM['t_anchor']:.2f}
MULTI-MATCH RULE: all candidates above the source threshold (no top-1), relative margin={FINAL_PRM['rel']}, exclusivity={FINAL_PRM['exclusive']}
SINGLETON RULE: empty list unless the best pair of the S1 reaches the anchor threshold

SUBMISSION
----------
matching_results.tsv: {OUT_MATCH}
candidate_pairs.tsv: {OUT_CAND}
VALIDATION STATUS: {'PASS' if SUBMISSION_PASS else 'FAIL'}
"""
print(REPORT)
display(EXP_TABLE)
display(pd.DataFrame(FINDINGS)[['finding', 'action']])

DOC = f"""# ML Challenge 2026: Business Entity Resolution Solution

**Team Name:** [fill in]
**Team Members:** [fill in]
**Submission Date:** {time.strftime('%Y-%m-%d')}

---

## 1. Executive Summary
Multi-pass, label-free blocking (exact keys + within-country TF-IDF retrieval{' + learned GPU encoder' if 'G' in SELECTED else ''}) feeds a
gradient-boosted pair classifier trained on hard (blocking) negatives{', re-ranked by a cross-source confirmation stage' if USE_STAGE2 else ''}; a decision rule tuned directly
for macro F0.5 (per-source thresholds, anchor/expansion singleton gate{', exclusivity of S2/S3 records' if FINAL_PRM['exclusive'] else ''}) produces the matches.
Hold-out macro F0.5 = {_hold['F05']:.4f}.

## 2. Methodology
### 2.1 Problem Analysis
""" + '\n'.join(f"- **{r['finding']}** — {r['evidence']}. → {r['action']}" for r in FINDINGS) + f"""

### 2.2 Solution Strategy
**Approach Type:** Blocking + Classifier (+ stacked reranker) + F0.5-optimised decision rule
**Core Innovation:** dependency-free Unicode transliteration + learned token maps; recall-driven greedy blocking union; anchor/expansion decision rule; cross-source confirmation.

## 3. Candidate Generation (Blocking)
- **Blocking keys used:** {', '.join(PASS_NAMES[p] + (f' (K={K_SEL[p]})' if K_SEL[p] else '') for p in SELECTED)}
- **Candidate pairs generated (test):** {SUB_STATS['total_candidate_pairs']:,} ({SUB_STATS['total_candidate_pairs'] / max(NQT, 1):.1f} per S1)
- **How true matches were kept:** passes chosen greedily by fit-fold recall; validation recall {FINAL_BLOCK['recall']:.4f}, reduction ratio {FINAL_BLOCK['reduction_ratio']:.6f}.

## 4. Matching Model
**Features used:** name/address equality at several normalisation levels, rapidfuzz ratios (Levenshtein, Jaro-Winkler, token-set/sort, partial),
IDF-weighted token Jaccard/cosine, char-3gram cosine, rare-token overlap, number/postal agreement & conflict, name/address commonness, noise flags,
within-S1 rank/gap/z-score context{', learned-encoder cosine' if USE_EMB else ''}{', stage-2 cross-source agreement' if USE_STAGE2 else ''}.
**Model type:** {FINAL_SCORER}
**Threshold selection method:** staged search maximising macro F0.5 on a tune fold of S1 entities; reported on an untouched hold fold.

## 5. Results & Error Analysis
- **F_0.5 Score (macro, hold):** {_hold['F05']:.4f} (precision {_hold['P_micro']:.4f}, recall {_hold['R_micro']:.4f})
- **Common false positives:** """ + ', '.join(f'{i} ({r["share_within_kind_%"]:.0f}%)' for i, r in ERR[ERR.kind == 'false positive'].head(4).iterrows()) + """
- **Common false negatives:** """ + ', '.join(f'{i} ({r["share_within_kind_%"]:.0f}%)' for i, r in ERR[ERR.kind == 'false negative'].head(4).iterrows()) + f"""

## 6. Conclusion
Blocking recall sets the ceiling ({ORACLE['hold']['F05']:.4f} oracle F0.5); hard-negative training and an F0.5-tuned set-level decision rule
convert it into {_hold['F05']:.4f}. Only competition data is used; no external lookup, geocoding or pretrained external models.

## Appendix — Experiment log
""" + EXP_MD
try:
    with open(os.path.join(CFG['WORK_DIR'], 'Documentation_template_filled.md'), 'w', encoding='utf-8') as fh:
        fh.write(DOC)
except Exception as e:
    print('documentation not written:', e)
print('Artifacts in', ART_DIR, ':', sorted(os.listdir(ART_DIR)))
print('Final files:', OUT_MATCH, OUT_CAND)